# SwarmNet: a learned drone swarm that keeps its radio network alive

**The problem.** A team of drones must survey every point on a 600 m × 600 m farm and bring every reading back to the base. Each drone can only talk to drones within radio range, readings hop drone-to-drone to reach the base, and drones can fail mid-mission. The number of drones is one setting, so the same code runs 5, 10, 20 or 30 drones.

**The method.** Every drone runs the same small neural network (graph attention over its radio neighbours), trained with multi-agent reinforcement learning (MAPPO). Each drone sees only local information and chooses one of three things:

| Choice | What it means |
|---|---|
| fly to a nearby point | survey one of its 6 nearest open points |
| head home | deliver the readings it carries, recharge, park |
| **relay** | hover where it is and pass messages along, so drones further out stay connected to the base |

**What it is compared with.**

| Method | Role |
|---|---|
| Central planner | an optimization model that sees everything and plans all routes (min-makespan vehicle routing). With unlimited radio and no failures it is **our central plan**, the reference every result is measured against. It is also run with the real radio limits. |
| Nearest-task rule | the simple baseline: each drone takes the closest open point. It runs inside the **same controller** as the learned swarm, so any difference comes from the decisions alone. |
| Learned swarm | the trained policy above. |

Every result is a **gap to our central plan** (% slower). Part 1 checks that plan before anything else.

**How to run:** Runtime → Run all. No GPU needed; free Colab is enough. `QUICK = True` is a preview (about 25 min, 2 farms per setting). `QUICK = False` is the reported study (about 35–75 min, 3 farms per setting, swarms up to 30 drones). With `SAVE_TO_DRIVE = True`, Colab asks for Google Drive access once at the start, and results are copied to Drive after every part, so a disconnect never loses finished work.

In [ ]:
# ---------- SETTINGS ----------
QUICK = False         # True: preview (~25 min). False: the reported study, sized for free Colab (~35-75 min).
USE_ORTOOLS = False   # True: OR-Tools polishes the central plan (much slower, rarely better)
SAVE_TO_DRIVE = True  # True: copy all results to Google Drive at the end (asks for permission once)

## Step 1 — Write the code package
Each cell below writes one file of the `swarmnet` package. Just run them.

If you edit a code cell here, re-run it and then **Runtime → Restart session** before running the rest.

In [ ]:
import os
for d in ["swarmnet", "results", "replays", "runs"]:
    os.makedirs(d, exist_ok=True)
print("folders ready:", os.getcwd())

In [ ]:
%%writefile swarmnet/__init__.py
"""SwarmNet: a drone survey network with limited radio, where a learned policy decides for each drone.

Modules
  config       all settings (the number of drones is one setting)
  world        the simulated farm: drones, points, base, radio links, battery, failures, relay hovering
  comm         neighbour-only messaging and lost-drone detection
  central      central planner (our optimization reference)
  routing      routing solvers: a built-in search, OR-Tools (optional), and an exact MILP for small cases
  swarm        swarm plumbing (and the auction swarm, kept for reference but not used in the notebook)
  learned      learned-swarm controller, the local observation, and the nearest-task rule
  rollout      rewards and advantages for training (no PyTorch needed)
  marl         graph-attention policy + MAPPO training (needs PyTorch)
  experiments  sweeps over radio range, failures and swarm size, plus validity checks
  plots        charts
  replay       HTML replay dashboard
"""
__version__ = "1.4.1"

In [ ]:
%%writefile swarmnet/config.py
"""All settings for one mission. Change a number here (or pass it in) to change the world.

The number of drones is a single setting: Config(n_drones=40) is all it takes to scale up.
"""
from dataclasses import dataclass, field, replace
from typing import List, Tuple


@dataclass
class Config:
    # --- the world -------------------------------------------------------------------
    n_drones: int = 10            # swarm size: change this to scale up or down
    n_tasks: int = 100            # survey points on the farm
    field_size: float = 600.0     # square field, metres per side
    base: Tuple[float, float] = (300.0, 0.0)  # base station: middle of the bottom edge

    # --- drones ----------------------------------------------------------------------
    speed: float = 8.0            # metres per second
    survey_time: float = 10.0     # seconds spent at each point taking the reading
    battery_range: float = 4000.0 # metres a full battery can fly
    battery_margin: float = 50.0  # metres kept in reserve when deciding to go home
    recharge_time: float = 30.0   # seconds on the pad to recharge

    # --- communication ---------------------------------------------------------------
    radio_range: float = 150.0    # metres: two nodes can talk if they are this close
    fail_detect_time: float = 90.0  # seconds of silence before a drone is presumed lost (trade-off: faster recovery vs false alarms)

    # --- failures --------------------------------------------------------------------
    fail_count: int = 0           # how many drones fail during the mission
    fail_time: float = 150.0      # when they fail (seconds after start)

    # --- simulation ------------------------------------------------------------------
    dt: float = 1.0               # seconds per simulation step
    max_time: float = 3000.0      # stop here even if the job is not finished
    seed: int = 0
    use_ortools: bool = False     # central planner: polish its plan with OR-Tools (slower; optional)
    trace_every: int = 2          # record a replay frame every N steps (0 = no replay)

    # --- relay (learned swarm) -------------------------------------------------------
    hover_drain: float = 4.0      # metres of battery used per second while hovering as a relay

    # --- auction swarm (CBBA) --------------------------------------------------------
    bundle_size: int = 5          # most tasks a drone can claim at once
    score_tau: float = 300.0      # time scale of the bid score (seconds)
    outbid_margin: float = 0.10   # a new bid must beat the old one by 10% to steal a task
    candidate_limit: int = 25     # only the nearest open tasks are considered when bidding
    rebuild_every: float = 3.0    # seconds between bundle rebuilds

    def with_(self, **kw) -> "Config":
        """Copy with some fields changed, e.g. cfg.with_(radio_range=60)."""
        return replace(self, **kw)

In [ ]:
%%writefile swarmnet/controller.py
"""The common shape every controller follows. The world calls these hooks."""


class Controller:
    name = "base"
    messages = 0

    def reset(self, world):
        """Called once when the mission starts."""

    def update(self, world):
        """Called every step before the drones move. Change world.queue[i] to give orders."""

    def on_complete(self, world, i, task):
        """Drone i has just surveyed `task` and now carries its reading."""

    def on_delivered(self, world, tasks):
        """These readings just reached the base."""

    def on_failure(self, world, ids):
        """Drones in `ids` have just failed. Only the central planner may use this directly;
        swarm controllers must notice failures on their own, through missing messages."""

In [ ]:
%%writefile swarmnet/world.py
"""The simulated farm: drones, survey points, base station, radio links, battery and failures.

The world only knows physics and the ground truth. Decisions (which drone goes where) come from a
controller: the central planner, the auction swarm, or the learned swarm. Every controller runs in
exactly the same world, so their results can be compared fairly.
"""
from __future__ import annotations

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

from .config import Config

# drone modes
IDLE, FLY, SURVEY, HOME, CHARGE, DEAD = range(6)
MODE_NAMES = ["idle", "fly", "survey", "home", "charge", "dead"]

# ground-truth task states
OPEN, HELD, DELIVERED = 0, 1, 2


class World:
    def __init__(self, cfg: Config, controller):
        self.cfg = cfg
        self.controller = controller
        self.reset()

    # ------------------------------------------------------------------ setup
    def reset(self):
        c = self.cfg
        self.rng = np.random.default_rng(c.seed)
        self.N, self.M = c.n_drones, c.n_tasks
        self.base = np.array(c.base, dtype=float)
        self.task_xy = self.rng.uniform(0, c.field_size, size=(self.M, 2))

        # drones start parked at the base
        self.pos = np.repeat(self.base[None, :], self.N, axis=0).astype(float)
        self.battery = np.full(self.N, c.battery_range)
        self.alive = np.ones(self.N, dtype=bool)
        self.mode = np.full(self.N, IDLE)
        self.home_reason = ["park"] * self.N
        self.queue = [[] for _ in range(self.N)]
        self.hold = np.zeros(self.N, dtype=bool)       # learned swarm can choose to hold position
        self.survey_left = np.zeros(self.N)
        self.survey_task = np.full(self.N, -1)
        self.charge_left = np.zeros(self.N)
        self.held = [set() for _ in range(self.N)]      # readings carried, not yet delivered
        self.flown = np.zeros(self.N)
        self.hover_time = 0.0                           # drone-seconds spent hovering as a relay

        self.task_state = np.full(self.M, OPEN)
        self.holders = [set() for _ in range(self.M)]
        self.delivered_time = np.full(self.M, np.nan)
        self.complete_count = np.zeros(self.M, dtype=int)

        self.t = 0.0
        self.steps = 0
        self.done = False
        self.makespan = None
        self.failed_yet = False
        self.events = []
        self.lost_tasks = set()
        self.fail_t = None
        self.first_redo = {}

        self.lam2_log, self.conn_log, self.basefrac_log = [], [], []
        self.frames = []
        self._update_comm()
        self.controller.reset(self)
        self._record(force=True)

    # ------------------------------------------------------------------ geometry helpers
    def node_positions(self):
        """Drone positions followed by the base (index N)."""
        return np.vstack([self.pos, self.base[None, :]])

    def node_alive(self):
        return np.append(self.alive, True)

    def dist_to_base(self, i):
        return float(np.linalg.norm(self.pos[i] - self.base))

    # ------------------------------------------------------------------ communication
    def _update_comm(self):
        P = self.node_positions()
        alive = self.node_alive()
        D = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=2)
        adj = (D <= self.cfg.radio_range) & alive[:, None] & alive[None, :]
        np.fill_diagonal(adj, False)
        self.adj = adj
        n_comp, labels = connected_components(csr_matrix(adj), directed=False)
        self.comp_labels = labels
        self.base_comp = (labels == labels[self.N]) & alive  # nodes that can reach the base

        # algebraic connectivity (Fiedler value) of the live network, base included
        idx = np.flatnonzero(alive)
        A = adj[np.ix_(idx, idx)].astype(float)
        L = np.diag(A.sum(1)) - A
        ev = np.linalg.eigvalsh(L) if len(idx) > 1 else np.array([0.0])
        self.lam2 = float(ev[1]) if len(ev) > 1 else 0.0
        self.connected = bool(np.all(labels[idx] == labels[idx[0]]))

    # ------------------------------------------------------------------ main loop
    def run(self):
        while not self.done:
            self.step()
        return self.metrics()

    def step(self):
        c = self.cfg
        if c.fail_count > 0 and not self.failed_yet and self.t >= c.fail_time:
            self._fail_random(c.fail_count)

        self.controller.update(self)
        for i in range(self.N):
            if self.alive[i]:
                if self.hold[i] and not self.queue[i] and self.mode[i] == IDLE:
                    self.hover_time += c.dt
                self._move(i)

        self._update_comm()
        self._deliver()
        self.lam2_log.append(self.lam2)
        self.conn_log.append(self.connected)
        a = self.alive.sum()
        self.basefrac_log.append(float(self.base_comp[: self.N].sum() / a) if a else 0.0)

        self.t += c.dt
        self.steps += 1
        if np.all(self.task_state == DELIVERED):
            self.done, self.makespan = True, self.t
        elif self.t >= c.max_time or not self.alive.any():
            self.done = True
        self._record(force=self.done)

    # ------------------------------------------------------------------ drone physics
    def _fly_toward(self, i, target):
        d = target - self.pos[i]
        dist = float(np.hypot(*d))
        step = self.cfg.speed * self.cfg.dt
        if dist <= step:
            self.pos[i] = target
            moved, arrived = dist, True
        else:
            self.pos[i] = self.pos[i] + d / dist * step
            moved, arrived = step, False
        self.battery[i] -= moved
        self.flown[i] += moved
        if self.battery[i] <= 0 and not arrived:
            self._kill(i, "battery ran out")
        return arrived

    def _move(self, i):
        c = self.cfg
        m = self.mode[i]
        if m == CHARGE:
            self.charge_left[i] -= c.dt
            if self.charge_left[i] <= 0:
                self.battery[i] = c.battery_range
                self.mode[i] = IDLE
            return
        if m == SURVEY:
            self.survey_left[i] -= c.dt
            if self.survey_left[i] <= 0:
                task = int(self.survey_task[i])
                if task in self.queue[i]:
                    self.queue[i].remove(task)
                self.survey_task[i] = -1
                self.mode[i] = IDLE
                self._complete(i, task)
            return
        if m == HOME and self.home_reason[i] == "charge":
            if self._fly_toward(i, self.base):
                self.mode[i] = CHARGE
                self.charge_left[i] = c.recharge_time
            return

        if self.queue[i]:
            task = self.queue[i][0]
            tgt = self.task_xy[task]
            need = (np.linalg.norm(self.pos[i] - tgt) + np.linalg.norm(tgt - self.base)
                    + c.battery_margin)
            if self.battery[i] < need:
                self.mode[i], self.home_reason[i] = HOME, "charge"
                if self._fly_toward(i, self.base):
                    self.mode[i] = CHARGE
                    self.charge_left[i] = c.recharge_time
                return
            self.mode[i] = FLY
            if self._fly_toward(i, tgt):
                self.mode[i] = SURVEY
                self.survey_left[i] = c.survey_time
                self.survey_task[i] = task
            return

        if self.hold[i]:
            # hovering in place as a radio relay: costs battery, so come home before it runs low
            if self.battery[i] < self.dist_to_base(i) + c.battery_margin + c.hover_drain * c.dt:
                self.hold[i] = False
                self.mode[i], self.home_reason[i] = HOME, "charge"
                if self._fly_toward(i, self.base):
                    self.mode[i] = CHARGE
                    self.charge_left[i] = c.recharge_time
                return
            self.battery[i] -= c.hover_drain * c.dt
            self.mode[i] = IDLE
            return
        if np.allclose(self.pos[i], self.base):
            self.mode[i] = IDLE
        else:
            self.mode[i], self.home_reason[i] = HOME, "park"
            if self._fly_toward(i, self.base):
                self.mode[i] = IDLE

    # ------------------------------------------------------------------ tasks and data
    def _complete(self, i, task):
        self.complete_count[task] += 1
        if self.fail_t is not None and task in self.lost_tasks and task not in self.first_redo:
            self.first_redo[task] = self.t
        if self.task_state[task] != DELIVERED:
            self.task_state[task] = HELD
            self.holders[task].add(i)
            self.held[i].add(task)
        self.controller.on_complete(self, i, task)

    def _deliver(self):
        new = []
        for i in np.flatnonzero(self.base_comp[: self.N]):
            if self.held[i]:
                for task in list(self.held[i]):
                    if self.task_state[task] != DELIVERED:
                        self.task_state[task] = DELIVERED
                        self.delivered_time[task] = self.t
                        new.append(task)
                    for h in self.holders[task]:
                        if h != i:
                            self.held[h].discard(task)
                    self.holders[task].clear()
                self.held[i].clear()
        if new:
            self.controller.on_delivered(self, new)

    # ------------------------------------------------------------------ failures
    def _kill(self, i, why):
        self.alive[i] = False
        self.mode[i] = DEAD
        self.queue[i] = []
        self.hold[i] = False
        self.survey_task[i] = -1
        lost = []
        for task in self.held[i]:
            self.holders[task].discard(i)
            if not self.holders[task] and self.task_state[task] == HELD:
                self.task_state[task] = OPEN
                lost.append(task)
        self.held[i].clear()
        self.events.append({"t": self.t, "drone": int(i), "what": why, "lost": len(lost)})
        return lost

    def _fail_random(self, k):
        self.failed_yet = True
        cand = np.flatnonzero(self.alive)
        k = min(k, len(cand))
        ids = self.rng.choice(cand, size=k, replace=False)
        self.fail_t = self.t
        for i in ids:
            plan = [x for x in self.queue[i] if self.task_state[x] == OPEN]
            lost = self._kill(i, "failure")
            self.lost_tasks.update(lost)
            self.lost_tasks.update(plan)
        self.controller.on_failure(self, [int(i) for i in ids])

    # ------------------------------------------------------------------ results
    def metrics(self):
        rec = None
        if self.fail_t is not None and self.lost_tasks:
            if all(t in self.first_redo for t in self.lost_tasks):
                rec = max(self.first_redo[t] for t in self.lost_tasks) - self.fail_t
        return {
            "method": getattr(self.controller, "name", "?"),
            "n_drones": self.N,
            "radio_range": self.cfg.radio_range,
            "fail_count": self.cfg.fail_count,
            "seed": self.cfg.seed,
            "finished": self.makespan is not None,
            "makespan": self.makespan,
            "time_used": self.t,
            "delivered_frac": float(np.mean(self.task_state == DELIVERED)),
            "total_distance": float(self.flown.sum()),
            "mean_lambda2": float(np.mean(self.lam2_log)) if self.lam2_log else 0.0,
            "connected_frac": float(np.mean(self.conn_log)) if self.conn_log else 0.0,
            "base_contact_frac": float(np.mean(self.basefrac_log)) if self.basefrac_log else 0.0,
            "repeat_surveys": int(np.maximum(self.complete_count - 1, 0).sum()),  # redone (lost or overlap)
            "messages": int(getattr(self.controller, "messages", 0)),
            "recovery_time": rec,
            "relay_seconds": float(self.hover_time),
            "readings_lost": int(sum(e["lost"] for e in self.events)),
        }

    # ------------------------------------------------------------------ replay
    def _record(self, force=False):
        k = self.cfg.trace_every
        if not k or (self.steps % k and not force):
            return
        drones = []
        for i in range(self.N):
            tgt = None
            if self.alive[i]:
                if self.mode[i] == HOME or self.mode[i] == CHARGE:
                    tgt = self.base.tolist()
                elif self.queue[i]:
                    tgt = self.task_xy[self.queue[i][0]].tolist()
            drones.append([round(float(self.pos[i, 0]), 1), round(float(self.pos[i, 1]), 1),
                           int(self.alive[i]), int(self.mode[i]), len(self.held[i]),
                           None if tgt is None else [round(tgt[0], 1), round(tgt[1], 1)],
                           round(float(self.battery[i] / self.cfg.battery_range), 3)])
        iu = np.argwhere(np.triu(self.adj, 1))
        self.frames.append({
            "t": round(self.t, 1),
            "d": drones,
            "l": iu.tolist(),
            "bc": np.flatnonzero(self.base_comp).tolist(),
            "ts": "".join(map(str, self.task_state.tolist())),
            "lam2": round(self.lam2, 3),
        })

    def trace(self):
        return {
            "meta": {
                "method": getattr(self.controller, "name", "?"),
                "n_drones": self.N, "n_tasks": self.M,
                "field": self.cfg.field_size, "base": self.base.tolist(),
                "radio_range": self.cfg.radio_range,
                "tasks": np.round(self.task_xy, 1).tolist(),
                "events": self.events,
                "metrics": self.metrics(),
            },
            "frames": self.frames,
        }

In [ ]:
%%writefile swarmnet/routing.py
"""Vehicle routing solvers used by the central planner.

The problem: K drones start at given points (possibly busy for a few seconds first), must visit every
task once, and end at the base. Objective: minimise the time the LAST drone finishes (makespan),
which spreads the work evenly instead of just shortening total distance.

Two solvers:
  * solve_ortools   Google OR-Tools (best quality; installed with `pip install ortools`)
  * solve_heuristic pure Python: cheapest balanced insertion + local search (always available)
plus solve_exact, an exact MILP (scipy) for small cases, used to measure how close the others get.
"""
from __future__ import annotations

import numpy as np


def route_time(route, start_xy, offset, tasks_xy, base, speed, survey, open_end=True):
    """Seconds for one drone: start offset + travel + surveys (+ flight back to base if not open_end).

    open_end=True matches the mission score when every reading can be relayed home at once, so the
    plan finishes at the last survey. This makes the central planner an optimistic best case."""
    t, p = offset, start_xy
    for j in route:
        q = tasks_xy[j]
        t += np.hypot(*(q - p)) / speed + survey
        p = q
    return t if open_end else t + np.hypot(*(base - p)) / speed


# ---------------------------------------------------------------------------- heuristic
def solve_heuristic(starts, offsets, tasks_xy, base, speed, survey, time_budget=2.0, open_end=True,
                    seed=0):
    """Large neighbourhood search: build routes by balanced insertion, then repeatedly remove a few
    points and re-insert them where they hurt the slowest drone least. Keeps the best plan found."""
    import time as _time
    rng = np.random.default_rng(seed)
    starts, tasks_xy, base = np.asarray(starts, float), np.asarray(tasks_xy, float), np.asarray(base, float)
    K, T = len(starts), len(tasks_xy)
    if T == 0 or K == 0:
        return [[] for _ in range(K)]
    pts = np.vstack([starts, tasks_xy, base[None, :]])
    D = np.hypot(*(pts[:, None, :] - pts[None, :, :]).transpose(2, 0, 1)) / speed
    BASE = K + T
    off = np.asarray(offsets, float)

    def r_time(k, r):
        if not r:
            return off[k]
        nodes = [k] + [K + j for j in r]
        t = off[k] + D[nodes[:-1], nodes[1:]].sum() + survey * len(r)
        return t if open_end else t + D[nodes[-1], BASE]

    def insert_all(routes, times, todo):
        for c in todo:
            cn = K + c
            best = None
            for k in range(K):
                r = routes[k]
                prev = np.array([k] + [K + j for j in r])
                if open_end:
                    nxt = np.array([K + j for j in r] + [-1])
                    delta = D[prev, cn] + survey
                    has = nxt >= 0
                    delta[has] += D[cn, nxt[has]] - D[prev[has], nxt[has]]
                else:
                    nxt = np.array([K + j for j in r] + [BASE])
                    delta = D[prev, cn] + D[cn, nxt] - D[prev, nxt] + survey
                q = int(np.argmin(delta))
                new_t = times[k] + delta[q]
                others = max([times[m] for m in range(K) if m != k], default=0.0)
                key = (max(new_t, others), delta[q])
                if best is None or key < best[0]:
                    best = (key, k, q, new_t)
            _, k, q, new_t = best
            routes[k].insert(q, c)
            times[k] = new_t

    def two_opt(routes, times, k):
        r = routes[k]
        improved = True
        while improved and len(r) > 2:
            improved = False
            for a_ in range(len(r) - 1):
                for b_ in range(a_ + 1, len(r)):
                    cand = r[:a_] + r[a_:b_ + 1][::-1] + r[b_ + 1:]
                    if r_time(k, cand) < r_time(k, r) - 1e-9:
                        r, improved = cand, True
        routes[k] = r
        times[k] = r_time(k, r)

    def key_of(times):
        return (max(times), sum(times))

    routes = [[] for _ in range(K)]
    times = [off[k] for k in range(K)]
    order = list(np.argsort(-np.hypot(*(tasks_xy - base).T)))
    insert_all(routes, times, order)
    for k in range(K):
        two_opt(routes, times, k)
    best_routes, best_times = [list(r) for r in routes], list(times)

    # a fixed number of attempts, never a clock: the same farm always gives the same plan,
    # on any computer (a wall-clock cut-off would give different plans on slower machines)
    max_iters = int(300 * time_budget)
    for it in range(max_iters):
        cur = [list(r) for r in best_routes]
        n_rm = int(rng.integers(2, max(3, min(15, T // 4 + 2))))
        mode = rng.integers(3)
        if mode == 0:     # random points
            removed = list(rng.choice(T, size=min(n_rm, T), replace=False))
        elif mode == 1:   # points near a random point
            c0 = int(rng.integers(T))
            removed = list(np.argsort(D[K + c0, K:K + T])[:n_rm])
        else:             # points from the slowest drone
            k_max = int(np.argmax(best_times))
            pool = cur[k_max]
            removed = list(rng.choice(pool, size=min(n_rm, len(pool)), replace=False)) if pool else []
        if not removed:
            continue
        rm = set(int(x) for x in removed)
        cur = [[j for j in r if j not in rm] for r in cur]
        cur_t = [r_time(k, cur[k]) for k in range(K)]
        rng.shuffle(removed)
        insert_all(cur, cur_t, [int(x) for x in removed])
        if max(cur_t) <= max(best_times) * 1.02:   # only polish promising plans
            for k in range(K):
                two_opt(cur, cur_t, k)
            if key_of(cur_t) < key_of(best_times):
                best_routes, best_times = cur, cur_t
    return [[int(j) for j in r] for r in best_routes]


# ---------------------------------------------------------------------------- OR-Tools
_ORTOOLS_WARNED = []


def plan_makespan(routes, starts, offsets, tasks_xy, base, speed, survey, open_end=True):
    """Finish time of the slowest drone for a set of routes."""
    return max((route_time(r, starts[k], offsets[k], tasks_xy, base, speed, survey, open_end)
                for k, r in enumerate(routes)), default=0.0)


def solve_ortools(starts, offsets, tasks_xy, base, speed, survey, solution_limit=300, safety_seconds=60,
                  open_end=True, initial_routes=None):
    """Min-makespan routing with OR-Tools. Returns routes, or None if OR-Tools is missing or fails.

    Objective: the time dimension's global span (latest finish - earliest start) = the makespan,
    with a tiny arc cost as tie-break. Two things make it reliable:
      * it is WARM-STARTED from `initial_routes` (the built-in search's plan), so it can only
        improve on a plan that already spreads the work over every drone;
      * it stops after a fixed number of solutions (`solution_limit`), not a wall-clock limit,
        so the same farm always gives the same plan. `safety_seconds` is only a backstop.
    """
    try:
        from ortools.constraint_solver import pywrapcp, routing_enums_pb2
    except Exception:
        return None
    try:
        starts, tasks_xy, base = np.asarray(starts, float), np.asarray(tasks_xy, float), np.asarray(base, float)
        K, T = len(starts), len(tasks_xy)
        if T == 0:
            return [[] for _ in range(K)]
        pts = np.vstack([starts, tasks_xy, base[None, :]])
        n = len(pts)
        end = n - 1
        SCALE = 10  # deciseconds: OR-Tools needs integers
        D = np.hypot(*(pts[:, None, :] - pts[None, :, :]).transpose(2, 0, 1))
        travel = np.rint(D / speed * SCALE).astype(np.int64)
        if open_end:
            travel[:, end] = 0   # the route ends at its last survey
        service = np.zeros(n, dtype=np.int64)
        service[K:K + T] = int(round(survey * SCALE))
        transit = (travel + service[:, None]).tolist()    # time from leaving i to arriving at j

        mgr = pywrapcp.RoutingIndexManager(n, K, list(range(K)), [end] * K)
        model = pywrapcp.RoutingModel(mgr)

        def time_cb(i, j):
            return transit[mgr.IndexToNode(i)][mgr.IndexToNode(j)]

        t_idx = model.RegisterTransitCallback(time_cb)
        model.SetArcCostEvaluatorOfAllVehicles(t_idx)            # tie-break only (coefficient 1)
        horizon = int(travel.max() * (T + 2) + service.sum() + max(offsets) * SCALE + 10)
        model.AddDimension(t_idx, 0, horizon, False, "Time")
        dim = model.GetDimensionOrDie("Time")
        for k in range(K):
            off = int(round(offsets[k] * SCALE))
            dim.CumulVar(model.Start(k)).SetRange(off, off)
        # span weight >> total arc cost, so the makespan always dominates
        dim.SetGlobalSpanCostCoefficient(int(10 * (T + K)))

        params = pywrapcp.DefaultRoutingSearchParameters()
        params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
        params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        params.solution_limit = int(solution_limit)
        params.time_limit.seconds = int(safety_seconds)

        if initial_routes is not None:
            model.CloseModelWithParameters(params)
            init = model.ReadAssignmentFromRoutes([[K + int(j) for j in r] for r in initial_routes], True)
            sol = model.SolveFromAssignmentWithParameters(init, params) if init is not None else None
        else:
            sol = model.SolveWithParameters(params)
        if sol is None:
            return None
        routes = []
        for k in range(K):
            i, r = sol.Value(model.NextVar(model.Start(k))), []
            while not model.IsEnd(i):
                r.append(mgr.IndexToNode(i) - K)
                i = sol.Value(model.NextVar(i))
            routes.append(r)
        return routes
    except Exception as e:  # an OR-Tools API difference must never break the study
        if not _ORTOOLS_WARNED:
            _ORTOOLS_WARNED.append(1)
            print(f"[routing] OR-Tools failed ({type(e).__name__}: {e}); using the built-in search only.")
        return None


def solve(starts, offsets, tasks_xy, base, speed, survey, prefer="ortools", time_limit=3, open_end=True):
    """Best available plan. Returns (routes, solver_name). Deterministic for a given farm.

    Always runs the built-in search first (fixed iteration count). If OR-Tools is installed and
    prefer == "ortools", it is warm-started from that plan and its answer is kept only if it is
    strictly faster. So adding OR-Tools can never make the plan worse.
    """
    starts = np.asarray(starts, float)
    base_routes = solve_heuristic(starts, offsets, tasks_xy, base, speed, survey, time_budget=time_limit,
                                  open_end=open_end)
    if prefer != "ortools" or len(tasks_xy) == 0:
        return base_routes, "heuristic"
    r = solve_ortools(starts, offsets, tasks_xy, base, speed, survey, solution_limit=int(100 * time_limit),
                      open_end=open_end, initial_routes=base_routes)
    if r is None:
        return base_routes, "heuristic"
    mk = lambda rr: plan_makespan(rr, starts, offsets, tasks_xy, base, speed, survey, open_end)
    if mk(r) < mk(base_routes) - 1e-9:
        return r, "ortools"
    return base_routes, "heuristic"


# ---------------------------------------------------------------------------- exact (small)
def solve_exact(tasks_xy, base, n_vehicles, speed, survey, time_limit=120, open_end=True):
    """Exact min-makespan routing from the base, as a MILP (scipy/HiGHS). Use for <= ~9 tasks.

    Returns (routes, makespan_seconds, proven_optimal)."""
    from scipy.optimize import LinearConstraint, Bounds, milp

    tasks_xy, base = np.asarray(tasks_xy, float), np.asarray(base, float)
    T, K = len(tasks_xy), n_vehicles
    pts = np.vstack([base[None, :], tasks_xy])
    n = T + 1
    D = np.hypot(*(pts[:, None, :] - pts[None, :, :]).transpose(2, 0, 1)) / speed
    if open_end:
        D[:, 0] = 0.0   # returning to the base costs nothing: the route ends at its last survey
    arcs = [(i, j) for i in range(n) for j in range(n) if i != j]
    A = len(arcs)
    nx = A * K
    iu = nx            # u_1..u_T  (MTZ order variables)
    iz = nx + T        # makespan
    nv = nx + T + 1

    def x(a, k):
        return k * A + a

    rows, lo, hi = [], [], []

    def add(coefs, l, h):
        row = np.zeros(nv)
        for idx, v in coefs:
            row[idx] += v
        rows.append(row); lo.append(l); hi.append(h)

    for j in range(1, n):   # each task entered exactly once
        add([(x(a, k), 1) for k in range(K) for a, (i, jj) in enumerate(arcs) if jj == j], 1, 1)
    for k in range(K):
        for v in range(n):  # flow conservation
            co = [(x(a, k), 1) for a, (i, j) in enumerate(arcs) if j == v]
            co += [(x(a, k), -1) for a, (i, j) in enumerate(arcs) if i == v]
            add(co, 0, 0)
        add([(x(a, k), 1) for a, (i, j) in enumerate(arcs) if i == 0], 0, 1)  # leave base once
        co = [(x(a, k), D[i, j] + (survey if j > 0 else 0)) for a, (i, j) in enumerate(arcs)]
        co.append((iz, -1))
        add(co, -np.inf, 0)                                                     # route time <= Z
    for k in range(K - 1):  # symmetry breaking: drone k's route is at least as long as drone k+1's
        co = [(x(a, k), D[i, j] + (survey if j > 0 else 0)) for a, (i, j) in enumerate(arcs)]
        co += [(x(a, k + 1), -(D[i, j] + (survey if j > 0 else 0))) for a, (i, j) in enumerate(arcs)]
        add(co, 0, np.inf)
    for a, (i, j) in enumerate(arcs):  # MTZ subtour elimination
        if i > 0 and j > 0:
            co = [(iu + i - 1, 1), (iu + j - 1, -1)] + [(x(a, k), T) for k in range(K)]
            add(co, -np.inf, T - 1)

    c = np.zeros(nv); c[iz] = 1
    integrality = np.zeros(nv); integrality[:nx] = 1
    lb = np.zeros(nv); ub = np.ones(nv)
    lb[iu:iu + T], ub[iu:iu + T] = 1, T
    ub[iz] = np.inf
    res = milp(c, constraints=LinearConstraint(np.array(rows), lo, hi), integrality=integrality,
               bounds=Bounds(lb, ub), options={"time_limit": time_limit})
    if res.x is None:
        return None, None, False
    xs = res.x
    routes = []
    for k in range(K):
        nxt = {i: j for a, (i, j) in enumerate(arcs) if xs[x(a, k)] > 0.5}
        r, cur = [], nxt.get(0)
        while cur is not None and cur != 0:
            r.append(cur - 1)
            cur = nxt.get(cur)
        routes.append(r)
    return routes, float(res.fun), res.status == 0   # status 0 = proven optimal

In [ ]:
%%writefile swarmnet/central.py
"""Central planner: one computer sees every drone and every point and plans all routes.

Real swarms cannot do this (it needs perfect information and a link to every drone), so it is used
only as the MEASURING STICK: the best result the swarm could hope for. It even replans instantly
when a drone fails, because it is told about failures directly.
"""
import numpy as np

from .controller import Controller
from .routing import solve
from .world import OPEN, SURVEY, CHARGE


class CentralPlanner(Controller):
    name = "central"

    def __init__(self, prefer=None, time_limit=3):
        """prefer=None follows the config (use_ortools); "ortools" or "heuristic" forces one."""
        self.prefer, self.time_limit = prefer, time_limit
        self.solver_used = None

    def reset(self, world):
        self.messages = 0
        self._plan(world)

    def on_failure(self, world, ids):
        self._plan(world)

    def _plan(self, world):
        c = world.cfg
        alive = np.flatnonzero(world.alive)
        busy = {int(world.survey_task[i]) for i in alive if world.mode[i] == SURVEY}
        todo = [j for j in range(world.M) if world.task_state[j] == OPEN and j not in busy]
        starts, offsets = [], []
        for i in alive:
            starts.append(world.pos[i].copy())
            if world.mode[i] == SURVEY:
                offsets.append(float(world.survey_left[i]))
            elif world.mode[i] == CHARGE:
                offsets.append(float(world.charge_left[i]))
            else:
                offsets.append(0.0)
        prefer = self.prefer or ("ortools" if getattr(c, "use_ortools", False) else "heuristic")
        routes, self.solver_used = solve(np.array(starts), offsets, world.task_xy[todo], world.base,
                                         c.speed, c.survey_time, prefer, self.time_limit)
        for k, i in enumerate(alive):
            plan = [todo[j] for j in routes[k]]
            if world.mode[i] == SURVEY:
                plan = [int(world.survey_task[i])] + plan
            world.queue[i] = plan

In [ ]:
%%writefile swarmnet/comm.py
"""Neighbour-only messaging shared by every swarm controller.

Each drone (and the base) keeps its own copy of a task table: for every survey point, who it thinks
has claimed it, with what bid, and whether it is done or delivered. Every second, each node swaps
tables ONLY with nodes in radio range, and keeps the better entry for each task. News therefore
travels one hop per second, like a message passed along a line of people.

Better = compared in this order:
  1. delivered beats everything (final)
  2. higher version (a release or a re-open bumps the version)
  3. done beats claimed beats open
  4. higher bid
  5. lower drone id (tie-break)
Drones also pass along heartbeats ("drone k was alive at time t"). If a drone that can reach the base
has not heard of drone k for a while, it presumes k lost and re-opens k's claimed tasks. Points k
surveyed but never delivered are re-opened only after a longer silence (three times as long), because
a drone that is simply out of range will usually come back and deliver them.
"""
import numpy as np

OPEN_S, CLAIMED_S, DONE_S = 0, 1, 2
LOCK = 1e6          # bids at or above this are "locked": the drone is already on its way


class Comm:
    def __init__(self, n_drones, n_tasks):
        n = n_drones + 1  # last node is the base
        self.N, self.M, self.n = n_drones, n_tasks, n
        self.ver = np.zeros((n, n_tasks), dtype=np.int64)
        self.st = np.zeros((n, n_tasks), dtype=np.int64)
        self.bid = np.zeros((n, n_tasks))
        self.win = np.full((n, n_tasks), -1, dtype=np.int64)
        self.dlv = np.zeros((n, n_tasks), dtype=bool)
        self.hb = np.zeros((n, n))
        self.messages = 0
        self._snap = None

    # ------------------------------------------------------------------ exchange
    def exchange(self, adj, alive_nodes, t, hb_period=5.0):
        for i in np.flatnonzero(alive_nodes):
            self.hb[i, i] = t
        old = (self.ver.copy(), self.st.copy(), self.bid.copy(), self.win.copy(), self.dlv.copy(),
               self.hb.copy())
        changed_before = self._changed_mask()
        has_nb = adj.any(1)
        beat = (int(round(t)) % int(hb_period) == 0)
        self.messages += int(np.sum(has_nb & alive_nodes & (changed_before | beat)))

        for i, j in np.argwhere(adj):
            self._merge_into(i, old, j)
            self.hb[i] = np.maximum(self.hb[i], old[5][j])
        self._snap = (self.ver.copy(), self.st.copy(), self.bid.copy(), self.win.copy(), self.dlv.copy())

    def _changed_mask(self):
        if self._snap is None:
            return np.ones(self.n, dtype=bool)
        v, s, b, w, d = self._snap
        return ((self.ver != v) | (self.st != s) | (self.bid != b) | (self.win != w)
                | (self.dlv != d)).any(1)

    def _merge_into(self, i, old, j):
        v, s, b, w, d = old[0][j], old[1][j], old[2][j], old[3][j], old[4][j]
        cv, cs, cb, cw, cd = self.ver[i], self.st[i], self.bid[i], self.win[i], self.dlv[i]
        better = (d & ~cd) | ((d == cd) & (
            (v > cv) | ((v == cv) & (
                (s > cs) | ((s == cs) & (
                    (b > cb) | ((b == cb) & (w < cw) & (w >= 0))))))))
        if better.any():
            self.ver[i, better] = v[better]
            self.st[i, better] = s[better]
            self.bid[i, better] = b[better]
            self.win[i, better] = w[better]
            self.dlv[i, better] = d[better]

    # ------------------------------------------------------------------ local edits
    def claim(self, i, j, bid):
        if not self.dlv[i, j]:
            self.st[i, j], self.bid[i, j], self.win[i, j] = CLAIMED_S, bid, i

    def release(self, i, j):
        if not self.dlv[i, j]:
            self.ver[i, j] += 1
            self.st[i, j], self.bid[i, j], self.win[i, j] = OPEN_S, 0.0, -1

    def mark_done(self, i, j):
        if not self.dlv[i, j]:
            self.st[i, j], self.win[i, j], self.bid[i, j] = DONE_S, i, LOCK * 10

    def mark_delivered(self, node, tasks):
        self.dlv[node, tasks] = True
        self.st[node, tasks] = DONE_S

    def reopen_lost(self, i, k, include_done=False):
        """Node i presumes drone k lost: re-open k's claimed tasks in i's table. With include_done,
        also re-open points k surveyed but never delivered (their readings died with k)."""
        m = (self.win[i] == k) & ((self.st[i] == CLAIMED_S) | (include_done & (self.st[i] == DONE_S))) & ~self.dlv[i]
        if m.any():
            self.ver[i, m] += 1
            self.st[i, m], self.bid[i, m], self.win[i, m] = OPEN_S, 0.0, -1

    def mine(self, i):
        return (self.win[i] == i) & (self.st[i] == CLAIMED_S) & ~self.dlv[i]

In [ ]:
%%writefile swarmnet/swarm.py
"""Decentralized swarm controllers: no boss, each drone decides from its own table.

AuctionSwarm  (CBBA, the Consensus-Based Bundle Algorithm)
    Every few seconds each drone looks at open points near it and bids for the ones it can do best.
    A bid is how much the point adds to the drone's own plan, scored so that finishing sooner is
    worth more. Bids spread one hop per second. If someone else holds a higher bid, the drone gives
    the point up, plus every point it added after it (those bids assumed it had the first one).

NearestTask   (simple baseline)
    Each drone claims the single nearest open point it knows about. Closer drones win ties.
"""
import numpy as np

from .comm import Comm, LOCK, OPEN_S, CLAIMED_S
from .controller import Controller
from .world import FLY, SURVEY, CHARGE


class SwarmBase(Controller):
    """Shared plumbing: messaging, failure detection, giving up lost claims."""

    name = "swarm"

    def reset(self, world):
        self.comm = Comm(world.N, world.M)
        self.next_think = np.arange(world.N) * 0.3  # stagger thinking so drones do not move in lockstep

    @property
    def messages(self):
        return self.comm.messages if hasattr(self, "comm") else 0

    def on_complete(self, world, i, task):
        self.comm.mark_done(i, task)

    def on_delivered(self, world, tasks):
        self.comm.mark_delivered(world.N, tasks)

    # ------------------------------------------------------------------
    def update(self, world):
        c, N, t = world.cfg, world.N, world.t
        comm = self.comm
        comm.exchange(world.adj, world.node_alive(), t)

        for i in np.flatnonzero(world.alive):
            # 1. presume silent drones lost, but only if I can still reach the base myself
            if t - comm.hb[i, N] <= 10.0:
                gap = t - comm.hb[i, :N]
                for k in np.flatnonzero(gap > c.fail_detect_time):
                    if k != i:
                        comm.reopen_lost(i, k, include_done=gap[k] > 3 * c.fail_detect_time)

            # 2. drop claims I have lost, and everything I planned after them
            q = world.queue[i]
            protected = int(world.survey_task[i]) if world.mode[i] == SURVEY else -1
            for idx, j in enumerate(q):
                if j == protected:
                    continue
                if not (comm.win[i, j] == i and comm.st[i, j] == CLAIMED_S and not comm.dlv[i, j]):
                    for later in q[idx + 1:]:
                        if later != protected and comm.win[i, later] == i and comm.st[i, later] == CLAIMED_S:
                            comm.release(i, later)
                    world.queue[i] = [x for x in q[:idx] ] + ([protected] if protected in q[idx:] else [])
                    break

            # 3. think: add new claims
            if t >= self.next_think[i]:
                self.next_think[i] = t + c.rebuild_every
                self.think(world, i)

            # 4. lock the point I am flying to, so nobody steals it mid-flight
            q = world.queue[i]
            if q and world.mode[i] in (FLY, SURVEY):
                j = q[0]
                if comm.win[i, j] == i and comm.bid[i, j] < LOCK:
                    comm.bid[i, j] += LOCK

    def think(self, world, i):
        raise NotImplementedError

    # helpers -------------------------------------------------------------
    def start_state(self, world, i):
        """Where and when drone i can start new work."""
        off = 0.0
        if world.mode[i] == SURVEY:
            off = float(world.survey_left[i])
        elif world.mode[i] == CHARGE:
            off = float(world.charge_left[i])
        return world.pos[i].copy(), off

    def candidates(self, world, i, allow_outbid):
        comm = self.comm
        free = (comm.st[i] == OPEN_S) & ~comm.dlv[i]
        if allow_outbid:
            free |= ((comm.st[i] == CLAIMED_S) & (comm.win[i] != i) & (comm.bid[i] < LOCK)
                     & ~comm.dlv[i])
        free[world.queue[i]] = False
        idx = np.flatnonzero(free)
        if len(idx) > world.cfg.candidate_limit:
            d = np.hypot(*(world.task_xy[idx] - world.pos[i]).T)
            idx = idx[np.argsort(d)[: world.cfg.candidate_limit]]
        return idx


class AuctionSwarm(SwarmBase):
    name = "auction"

    def think(self, world, i):
        c, comm = world.cfg, self.comm
        tau, v, s = c.score_tau, c.speed, c.survey_time
        while len(world.queue[i]) < c.bundle_size:
            cand = self.candidates(world, i, allow_outbid=True)
            if len(cand) == 0:
                return
            path = world.queue[i]
            p0, off = self.start_state(world, i)
            pts = [p0] + [world.task_xy[j] for j in path]
            done_t, t_cur = [], off           # finish time of each planned point
            for a in range(len(path)):
                t_cur += np.hypot(*(pts[a + 1] - pts[a])) / v + s
                done_t.append(t_cur)
            done_t = np.array(done_t)
            C = world.task_xy[cand]
            best_gain = np.full(len(cand), -np.inf)
            best_pos = np.zeros(len(cand), dtype=int)
            for q in range(len(path) + 1):
                P = pts[q]
                t_prev = off if q == 0 else done_t[q - 1]
                d_in = np.hypot(*(C - P).T)
                gain = np.exp(-(t_prev + d_in / v + s) / tau)
                if q < len(path):
                    nxt = pts[q + 1]
                    delta = (d_in + np.hypot(*(C - nxt).T) - np.hypot(*(nxt - P))) / v + s
                    after = done_t[q:]
                    gain = gain + (np.exp(-(after[None, :] + delta[:, None]) / tau)
                                   - np.exp(-after[None, :] / tau)).sum(1)
                better = gain > best_gain
                best_gain[better], best_pos[better] = gain[better], q
            current = comm.bid[i, cand] * np.where(comm.st[i, cand] == CLAIMED_S, 1 + c.outbid_margin, 1.0)
            ok = best_gain > current + 1e-9
            if not ok.any():
                return
            k = int(np.argmax(np.where(ok, best_gain, -np.inf)))
            j = int(cand[k])
            comm.claim(i, j, float(best_gain[k]))
            path.insert(int(best_pos[k]), j)


class NearestTask(SwarmBase):
    name = "nearest"

    def think(self, world, i):
        if world.queue[i]:
            return
        cand = self.candidates(world, i, allow_outbid=False)
        if len(cand) == 0:
            return
        d = np.hypot(*(world.task_xy[cand] - world.pos[i]).T)
        k = int(np.argmin(d))
        self.comm.claim(i, int(cand[k]), 1000.0 / (1.0 + d[k]))
        world.queue[i] = [int(cand[k])]

In [ ]:
%%writefile swarmnet/learned.py
"""Learned swarm: each drone's choice comes from a policy (a neural network after training).

A drone asks its policy for a decision whenever it has nothing to do. It sees ONLY local information:
  * itself:      position, battery, readings carried, whether it can reach the base, whether it is relaying
  * nearby points: the K nearest points its own table says are still open
  * neighbours:  drones within radio range (what they broadcast: position, battery, busy or not)
  * the base:    where it is and whether it is in range
It then picks one of three kinds of action:
  * fly to one of the K points and survey it
  * head home (deliver the readings it carries, recharge, park)
  * RELAY: hover where it is and pass messages along, so drones further out stay connected to the
    base and their readings reach home without flying back. The nearest-task rule never does this.

The observation has the same shape whatever the swarm size: K points and up to M neighbours, with
masks for missing ones. That is what lets a policy trained on 10 drones run on 20 or 30.

Fairness: the nearest-task rule (`nearest_policy`) runs inside this SAME controller, so a difference
between "learned" and "nearest" comes from the decisions alone, not from different plumbing.
"""
import numpy as np

from .comm import LOCK, OPEN_S
from .swarm import SwarmBase
from .world import IDLE, HOME, CHARGE, SURVEY

K_TASKS = 6        # points offered to the policy
M_NEIGH = 8        # neighbours the policy can see
SELF_DIM, TASK_DIM, NB_DIM, BASE_DIM = 9, 5, 6, 4
HOME_ACTION = K_TASKS            # head home
RELAY_ACTION = K_TASKS + 1       # hover here as a radio relay
N_ACTIONS = K_TASKS + 2
RETHINK = 15.0                   # seconds before a drone that chose "home" or "relay" decides again


def build_obs(world, comm, i):
    """Local observation for drone i. Returns a dict of fixed-shape arrays."""
    c, L, R, N = world.cfg, world.cfg.field_size, world.cfg.radio_range, world.N
    p = world.pos[i]
    db = np.linalg.norm(p - world.base)
    contact = float(world.t - comm.hb[i, N] <= 3.0)
    nb = np.flatnonzero(world.adj[i, :N])
    progress = float((comm.dlv[i] | (comm.st[i] == 2)).mean())
    self_f = np.array([p[0] / L, p[1] / L, world.battery[i] / c.battery_range, len(world.held[i]) / 5.0,
                       db / L, contact, len(nb) / 5.0, progress, float(world.hold[i])], dtype=np.float32)

    open_ = np.flatnonzero((comm.st[i] == OPEN_S) & ~comm.dlv[i])
    task_f = np.zeros((K_TASKS, TASK_DIM), dtype=np.float32)
    task_m = np.zeros(K_TASKS, dtype=bool)
    task_ids = np.full(K_TASKS, -1)
    if len(open_):
        d = np.hypot(*(world.task_xy[open_] - p).T)
        order = np.argsort(d)[:K_TASKS]
        for s, o in enumerate(order):
            j = open_[o]
            q = world.task_xy[j]
            dtb = np.linalg.norm(q - world.base)
            task_f[s] = [(q[0] - p[0]) / L, (q[1] - p[1]) / L, d[o] / L, dtb / L, float(dtb <= R)]
            task_m[s] = True
            task_ids[s] = j

    nb_f = np.zeros((M_NEIGH, NB_DIM), dtype=np.float32)
    nb_m = np.zeros(M_NEIGH, dtype=bool)
    if len(nb):
        dn = np.hypot(*(world.pos[nb] - p).T)
        for s, o in enumerate(np.argsort(dn)[:M_NEIGH]):
            k = nb[o]
            q = world.pos[k]
            nb_f[s] = [(q[0] - p[0]) / R, (q[1] - p[1]) / R, world.battery[k] / c.battery_range,
                       len(world.held[k]) / 5.0, float(not world.queue[k]), float(world.hold[k])]
            nb_m[s] = True

    base_f = np.array([(world.base[0] - p[0]) / L, (world.base[1] - p[1]) / L, db / L, float(db <= R)],
                      dtype=np.float32)
    return {"self": self_f, "tasks": task_f, "task_mask": task_m, "nb": nb_f, "nb_mask": nb_m,
            "base": base_f, "task_ids": task_ids}


def global_features(world):
    """Extra information for the training critic only (never used by drones in the field)."""
    return np.array([world.t / 600.0, float(np.mean(world.task_state == 2)),
                     float(np.mean(world.task_state >= 1)), world.alive.mean()], dtype=np.float32)


def stack(obs_list):
    keys = ["self", "tasks", "task_mask", "nb", "nb_mask", "base"]
    return {k: np.stack([o[k] for o in obs_list]) for k in keys}


class LearnedSwarm(SwarmBase):
    """policy(batch_obs, world, ids, obs) -> array of action indices, one per deciding drone."""

    name = "learned"

    def __init__(self, policy, name=None, recorder=None):
        self.policy = policy
        self.recorder = recorder      # used during training to store decisions
        if name:
            self.name = name

    def reset(self, world):
        super().reset(world)
        self.wait_until = np.zeros(world.N)
        if self.recorder is not None:
            self.recorder.start(world)

    def think(self, world, i):
        pass  # decisions are made together in decide(), not one by one

    def update(self, world):
        if self.recorder is not None:
            self.recorder.tick(world)
        super().update(world)
        self.decide(world)

    def decide(self, world):
        need = [i for i in np.flatnonzero(world.alive)
                if not world.queue[i] and world.mode[i] in (IDLE, HOME)
                and not (world.mode[i] == HOME and world.home_reason[i] == "charge")
                and world.t >= self.wait_until[i]]
        if not need:
            return
        obs = [build_obs(world, self.comm, i) for i in need]
        acts = np.asarray(self.policy(stack(obs), world, need, obs), dtype=int)
        for i, o, a in zip(need, obs, acts):
            if a < K_TASKS and o["task_mask"][a]:
                j = int(o["task_ids"][a])
                d = float(np.linalg.norm(world.task_xy[j] - world.pos[i]))
                self.comm.claim(i, j, LOCK + 1000.0 / (1.0 + d))   # closer drone wins a clash
                world.queue[i] = [j]
                world.hold[i] = False
            elif a == RELAY_ACTION:
                world.queue[i] = []
                world.hold[i] = True                                # hover here (see World._move)
                self.wait_until[i] = world.t + RETHINK
            else:
                world.queue[i] = []
                world.hold[i] = False                               # head home
                self.wait_until[i] = world.t + RETHINK


# ------------------------------------------------------------------ simple fixed policies
def nearest_policy(batch, world, ids, obs):
    """The nearest-task rule: always take the nearest open point; go home if none. Never relays."""
    return np.where(batch["task_mask"][:, 0], 0, HOME_ACTION)


def make_random_policy(seed=0, relay_prob=0.1):
    rng = np.random.default_rng(seed)

    def pol(batch, world, ids, obs):
        out = []
        for m in batch["task_mask"]:
            if rng.random() < relay_prob:
                out.append(RELAY_ACTION)
                continue
            valid = np.append(np.flatnonzero(m), HOME_ACTION)
            out.append(rng.choice(valid))
        return np.array(out)
    return pol

In [ ]:
%%writefile swarmnet/rollout.py
"""Training bookkeeping that does not need PyTorch: rewards, decision records and advantages.

Team reward, given every second:
    +10 * (new readings delivered to base) / (number of points)   reward for getting data home
    +5  * (change in the share of points surveyed)                 credit the moment a survey is done
    -1/300 per second                                              pressure to finish sooner
    +CONNECT_BONUS/300 * (share of live drones linked to the base) small reward for a connected team
    -10 * (share of readings still missing) if time runs out       penalty for an unfinished job

The survey term is "potential-based shaping": it is the change of a score (share surveyed), so it adds
up to the same total whatever the drones do and cannot change which plan is best. It only gives the
credit earlier: a choice that gets a survey done sooner is rewarded at once instead of at the very end.
A lost reading takes its survey credit back (the share surveyed goes down).
"""
import numpy as np

from .learned import build_obs, global_features

TIME_SCALE = 300.0
SURVEY_SHAPING = 5.0
CONNECT_BONUS = 0.5   # at most half the size of the time penalty, so finishing sooner still matters more


def critic_state(world, comm):
    """Per-drone features for the training critic (all live drones) plus global features."""
    feats = []
    for i in np.flatnonzero(world.alive):
        o = build_obs(world, comm, i)
        feats.append(np.concatenate([o["self"], o["base"]]))
    if not feats:
        feats = [np.zeros(12, dtype=np.float32)]
    return np.stack(feats).astype(np.float32), global_features(world)


class Recorder:
    def __init__(self):
        self.episodes = []

    def start(self, world):
        self.cur = {"steps": [], "rewards": [], "n_tasks": world.M}
        self.prev_delivered = 0
        self.prev_surveyed = 0.0

    def tick(self, world):
        d = int(np.sum(world.task_state == 2))
        sv = float(np.mean(world.task_state >= 1))
        alive = world.alive.sum()
        linked = float(world.base_comp[: world.N].sum() / alive) if alive else 0.0
        r = (10.0 * (d - self.prev_delivered) / world.M
             + SURVEY_SHAPING * (sv - self.prev_surveyed)
             - world.cfg.dt / TIME_SCALE
             + CONNECT_BONUS * linked * world.cfg.dt / TIME_SCALE)
        self.prev_delivered, self.prev_surveyed = d, sv
        self.cur["rewards"].append((world.t, r))

    def add_decision(self, t, batch, actions, logp, cstate):
        self.cur["steps"].append({"t": t, "obs": batch, "act": np.asarray(actions),
                                  "logp": np.asarray(logp, dtype=np.float32), "cstate": cstate})

    def finish(self, world):
        self.tick(world)
        if not world.makespan:
            missing = 1.0 - float(np.mean(world.task_state == 2))
            self.cur["rewards"].append((world.t, -10.0 * missing))
        self.cur["metrics"] = world.metrics()
        self.cur["return"] = float(sum(r for _, r in self.cur["rewards"]))
        self.episodes.append(self.cur)


def segment_rewards(ep, gamma):
    """Discounted reward collected between consecutive decision steps, and the discount to the next."""
    steps, rew = ep["steps"], ep["rewards"]
    times = [s["t"] for s in steps]
    R = np.zeros(len(steps))
    disc = np.zeros(len(steps))
    k = 0
    for (t, r) in rew:
        while k + 1 < len(times) and t >= times[k + 1]:
            k += 1
        if t >= times[0]:
            R[k] += (gamma ** (t - times[k])) * r
    for k in range(len(steps) - 1):
        disc[k] = gamma ** (times[k + 1] - times[k])
    return R, disc


def gae(R, disc, V, lam):
    """Generalised advantage estimation over decision steps with uneven time gaps."""
    A = np.zeros(len(R))
    last = 0.0
    for k in range(len(R) - 1, -1, -1):
        v_next = V[k + 1] if k + 1 < len(R) else 0.0
        delta = R[k] + disc[k] * v_next - V[k]
        last = delta + disc[k] * lam * last
        A[k] = last
    return A, A + V


def discounted_returns(R, disc):
    """Return-to-go at every decision step (used to pre-train the critic)."""
    G = np.zeros(len(R))
    run = 0.0
    for k in range(len(R) - 1, -1, -1):
        run = R[k] + disc[k] * run
        G[k] = run
    return G

In [ ]:
%%writefile swarmnet/marl.py
"""Learned swarm: graph-attention policy trained with MAPPO (needs PyTorch; Colab has it).

Networks
  Actor  (runs on every drone, shared weights)
         encodes itself, its neighbours and the base; ATTENTION decides how much to listen to each
         neighbour; then it scores each of the K nearby open points, "head home", and "relay"
         (hover here and pass messages to the base). Works for any number of neighbours and any
         swarm size.
  Critic (training only)
         encodes every live drone, averages and max-pools them, adds global progress, and predicts
         how well the rest of the mission will go. Pooling makes it work for any swarm size too.

Training (MAPPO = multi-agent PPO, centralised training, decentralised execution)
  0. WARM START (behaviour cloning): the actor first copies the nearest-task rule on recorded
     missions. The rule never relays, so the relay choice is left out of the copying and then
     given a small starting probability (RELAY_START), so reinforcement learning can try it.
  1. collect missions on HARD settings (short radio, drone failures: where the rule is weak) ->
     score each decision with the team reward (advantage) -> nudge the policy towards
     better-than-expected decisions, with PPO's clipping to keep each update small.
  A curriculum starts with 3 drones, then 5, then 10. The best checkpoint is kept, judged on
  validation farms that are never used for the results; the warm start is the first candidate.
"""
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn

from .config import Config
from .learned import (BASE_DIM, K_TASKS, NB_DIM, SELF_DIM, TASK_DIM, RELAY_ACTION, LearnedSwarm,
                      nearest_policy)
from .rollout import Recorder, critic_state, discounted_returns, gae, segment_rewards
from .world import World

VALUE_SCALE = 10.0                       # the critic predicts return / VALUE_SCALE
VALIDATION_SEEDS = range(2000, 2010)     # 10 farms used to PICK the best checkpoint (never in results)
VALIDATION_SETTINGS = ((100.0, 1), (150.0, 0))   # (radio range m, drones that fail): one hard, one easy
RELAY_START = 0.05                       # starting probability of "relay" after the warm start


def step(opt, loss, params, max_norm):
    """One optimiser step with its OWN gradient clip, so one network cannot shrink the other's update."""
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(params, max_norm)
    opt.step()


def mlp(i, h):
    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())


class Actor(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.h = h
        self.self_enc, self.nb_enc, self.base_enc = mlp(SELF_DIM, h), mlp(NB_DIM, h), mlp(BASE_DIM, h)
        self.q, self.k, self.v = nn.Linear(h, h), nn.Linear(h, h), nn.Linear(h, h)
        self.ctx = mlp(3 * h, h)
        self.task_enc = mlp(TASK_DIM, h)
        self.task_head = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 1))
        self.home_head = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
        self.relay_head = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))

    def forward(self, o):
        hs = self.self_enc(o["self"])                                  # (B, h)
        hn = self.nb_enc(o["nb"])                                      # (B, M, h)
        nb_mask = o["nb_mask"]                                         # (B, M) bool
        score = (self.q(hs).unsqueeze(1) * self.k(hn)).sum(-1) / math.sqrt(self.h)
        score = score.masked_fill(~nb_mask, -1e9)
        alpha = torch.softmax(score, dim=-1) * nb_mask.float()         # no neighbours -> all zero
        msg = (alpha.unsqueeze(-1) * self.v(hn)).sum(1)                # (B, h)
        c = self.ctx(torch.cat([hs, msg, self.base_enc(o["base"])], -1))
        ht = self.task_enc(o["tasks"])                                 # (B, K, h)
        tl = self.task_head(torch.cat([c.unsqueeze(1).expand(-1, ht.shape[1], -1), ht], -1)).squeeze(-1)
        tl = tl.masked_fill(~o["task_mask"], -1e9)
        logits = torch.cat([tl, self.home_head(c), self.relay_head(c)], -1)   # (B, K+2)
        return logits, alpha


class Critic(nn.Module):
    def __init__(self, h=64, agent_dim=SELF_DIM + BASE_DIM, glob_dim=4):
        super().__init__()
        self.enc = mlp(agent_dim, h)
        self.head = nn.Sequential(nn.Linear(2 * h + glob_dim, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(),
                                  nn.Linear(h, 1))

    def forward(self, feats, mask, glob):
        e = self.enc(feats)                                            # (B, N, h)
        m = mask.unsqueeze(-1).float()
        mean = (e * m).sum(1) / m.sum(1).clamp(min=1.0)
        mx = e.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values
        mx = torch.where(mask.any(1, keepdim=True), mx, torch.zeros_like(mx))
        return self.head(torch.cat([mean, mx, glob], -1)).squeeze(-1)


# ------------------------------------------------------------------ helpers
def to_t(batch, device):
    out = {}
    for k in ["self", "tasks", "nb", "base"]:
        out[k] = torch.as_tensor(batch[k], dtype=torch.float32, device=device)
    for k in ["task_mask", "nb_mask"]:
        out[k] = torch.as_tensor(batch[k], dtype=torch.bool, device=device)
    return out


def pad_states(cstates, device):
    n = max(f.shape[0] for f, _ in cstates)
    F = np.zeros((len(cstates), n, cstates[0][0].shape[1]), dtype=np.float32)
    Mk = np.zeros((len(cstates), n), dtype=bool)
    G = np.stack([g for _, g in cstates]).astype(np.float32)
    for b, (f, _) in enumerate(cstates):
        F[b, : len(f)] = f
        Mk[b, : len(f)] = True
    return (torch.as_tensor(F, device=device), torch.as_tensor(Mk, device=device),
            torch.as_tensor(G, device=device))


def make_policy(actor, device="cpu", greedy=False, recorder=None, seed=None):
    """Wrap the actor as a policy the LearnedSwarm controller can call.

    greedy=False (default) samples an action from the policy. A purely greedy policy can get stuck
    (a drone parked at base picks "stay home" forever). Sampling with a fixed seed is reproducible.
    """
    gen = None
    if seed is not None:
        gen = torch.Generator(device=device)
        gen.manual_seed(int(seed))

    def pol(batch, world, ids, obs):
        with torch.no_grad():
            logits, _ = actor(to_t(batch, device))
            dist = torch.distributions.Categorical(logits=logits)
            if greedy:
                a = logits.argmax(-1)
            else:
                a = torch.multinomial(torch.softmax(logits, -1), 1, generator=gen).squeeze(-1)
            lp = dist.log_prob(a)
        a_np = a.cpu().numpy()
        if recorder is not None:
            recorder.add_decision(world.t, batch, a_np, lp.cpu().numpy(),
                                  critic_state(world, world.controller.comm))
        return a_np
    return pol


def load_policy(path, device="cpu", greedy=False, seed=0):
    ck = torch.load(path, map_location=device, weights_only=False)
    actor = Actor(ck.get("hidden", 64)).to(device)
    actor.load_state_dict(ck["actor"])
    actor.eval()
    return make_policy(actor, device, greedy=greedy, seed=seed)


# ------------------------------------------------------------------ warm start
def collect_teacher(rng, base_cfg, sizes=(3, 5, 10), episodes=30, teacher=nearest_policy):
    """Run missions with a fixed rule and record every decision (observation + the rule's action)."""
    rec = Recorder()

    def pol(batch, world, ids, obs):
        a = np.asarray(teacher(batch, world, ids, obs))
        rec.add_decision(world.t, batch, a, np.zeros(len(a), np.float32), critic_state(world, world.controller.comm))
        return a

    for e in range(episodes):
        w = World(sample_cfg(rng, sizes[e % len(sizes)], base_cfg), LearnedSwarm(pol, recorder=rec))
        w.run()
        rec.finish(w)
    return rec.episodes


def _no_relay(logits):
    """The rule never relays, so the relay choice is left out while copying it."""
    out = logits.clone()
    out[:, RELAY_ACTION] = -1e9
    return out


def calibrate_relay(actor, batch, device, p=RELAY_START):
    """Shift the relay head's bias so that, on the recorded decisions, "relay" starts with average
    probability about p. Without this the copied policy would never try relaying."""
    with torch.no_grad():
        lg, _ = actor(to_t(batch, device))
        others = torch.logsumexp(_no_relay(lg), -1)
        base = lg[:, RELAY_ACTION]
        mean_p = lambda s: float(torch.sigmoid(base + s - others).mean())
        lo, hi = -50.0, 50.0                      # find the bias shift that gives mean probability p
        for _ in range(60):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if mean_p(mid) < p else (lo, mid)
        actor.relay_head[-1].bias += (lo + hi) / 2
        lg2, _ = actor(to_t(batch, device))
        return float(torch.softmax(lg2, -1)[:, RELAY_ACTION].mean())


def pretrain_bc(actor, critic, opt_a, opt_c, episodes, gamma=0.995, max_epochs=60, target=0.98, minibatch=256,
                device="cpu", rng=None, log=print):
    """Behaviour cloning. The actor learns to copy the rule (cross-entropy, relay left out); the critic
    learns how those missions went. Runs until the actor matches the rule on `target` of the
    decisions or `max_epochs` is reached. Returns (match rate, relay start probability)."""
    rng = rng or np.random.default_rng(0)
    obs_rows, acts, cst, rets = [], [], [], []
    for ep in episodes:
        if not ep["steps"]:
            continue
        R, disc = segment_rewards(ep, gamma)
        G = discounted_returns(R, disc)
        for k, st in enumerate(ep["steps"]):
            obs_rows.append(st["obs"]); acts.append(st["act"]); cst.append(st["cstate"]); rets.append(G[k])
    batch = {k: np.concatenate([o[k] for o in obs_rows]) for k in obs_rows[0]}
    acts_np = np.concatenate(acts).astype(np.int64)
    rets_np = np.asarray(rets, dtype=np.float32) / VALUE_SCALE
    n_dec, n_st = len(acts_np), len(cst)
    forced = float(np.mean(~batch["task_mask"][:, 0]))

    def match_all():
        actor.eval()
        hit = 0
        with torch.no_grad():
            for b0 in range(0, n_dec, 1024):
                sl = slice(b0, b0 + 1024)
                lg, _ = actor(to_t({k: v[sl] for k, v in batch.items()}, device))
                hit += int((_no_relay(lg).argmax(-1).cpu().numpy() == acts_np[sl]).sum())
        actor.train()
        return hit / max(1, n_dec)

    log(f"warm start: {n_dec} recorded decisions ({forced:.0%} have only one choice, 'head home')")
    actor.train()
    m = match_all()
    for ep_i in range(max_epochs):
        perm = rng.permutation(n_dec)
        tot = 0.0
        for b0 in range(0, n_dec, minibatch):
            idx = perm[b0:b0 + minibatch]
            logits, _ = actor(to_t({k: v[idx] for k, v in batch.items()}, device))
            a = torch.as_tensor(acts_np[idx], device=device)
            ce = -torch.distributions.Categorical(logits=_no_relay(logits)).log_prob(a).mean()
            step(opt_a, ce, actor.parameters(), 1.0)
            sidx = rng.choice(n_st, size=min(minibatch, n_st), replace=False)
            F, Mk, G = pad_states([cst[j] for j in sidx], device)
            vl = ((critic(F, Mk, G) - torch.as_tensor(rets_np[sidx], device=device)) ** 2).mean()
            step(opt_c, vl, critic.parameters(), 1.0)
            tot += float(ce.item()) * len(idx)
        m = match_all()
        if ep_i % 5 == 4 or m >= target:
            log(f"warm start epoch {ep_i + 1}: imitation loss {tot / n_dec:.3f}, matches the rule {m:.1%}")
        if m >= target:
            break
    if m < 0.95:
        log(f"WARNING: the warm start matches the rule only {m:.1%} (< 95%). Learning results are not meaningful.")
    p_relay = calibrate_relay(actor, batch, device)
    log(f"relay starts at {p_relay:.1%} average probability, so reinforcement learning can try it")
    return m, p_relay


# ------------------------------------------------------------------ training
def default_curriculum(it, total):
    """Which swarm size to train on: 3 drones for the first 20%, then 5, then 10 from halfway."""
    if it < 0.2 * total:
        return 3
    if it < 0.5 * total:
        return 5
    return 10


def sample_cfg(rng, n_drones, base_cfg, hard=True):
    """A random training farm. hard=True favours short radio and failures, where the rule is weak."""
    radios = [80, 100, 120, 150] if hard else [80, 120, 150, 200]
    fails = ([0, 1, 1, 2] if hard else [0, 0, 1, 2]) if n_drones >= 5 else [0, 0, 1]
    return base_cfg.with_(
        n_drones=n_drones,
        n_tasks=10 * n_drones,
        radio_range=float(rng.choice(radios)),
        fail_count=int(rng.choice(fails)),
        seed=int(rng.integers(1_000_000)),
        trace_every=0,
    )


def evaluate_policy(policy, n_drones=10, radio_range=150, fail_count=0, seeds=VALIDATION_SEEDS, base_cfg=None,
                    name="learned"):
    """Runs the policy on the given farm seeds (default: the validation farms, separate from the
    results' farms 0-4, so picking a checkpoint never peeks at the reported results)."""
    base_cfg = base_cfg or Config()
    out = []
    for s in seeds:
        cfg = base_cfg.with_(n_drones=n_drones, n_tasks=10 * n_drones, radio_range=radio_range,
                             fail_count=fail_count, seed=int(s), trace_every=0)
        out.append(World(cfg, LearnedSwarm(policy, name=name)).run())
    return out


def score_of(ev):
    """Higher is better: 10 per mission if every reading arrives, minus 1 per 300 s taken."""
    return float(np.mean([10 * e["delivered_frac"] - e["time_used"] / 300 for e in ev]))


def validation_score(policy, n_drones, base_cfg):
    return float(np.mean([score_of(evaluate_policy(policy, n_drones=n_drones, radio_range=r, fail_count=f,
                                                   base_cfg=base_cfg))
                          for r, f in VALIDATION_SETTINGS]))


def train(iterations=150, episodes_per_iter=6, hidden=64, lr_actor=1e-4, lr_critic=3e-4, gamma=0.995, lam=0.95,
          clip=0.2, epochs=4, minibatch=256, ent_coef=0.01, curriculum=default_curriculum, hard=True,
          save_path="runs/learned_policy.pt", device=None, seed=0, base_cfg=None, eval_every=10,
          bc_episodes=40, bc_max_epochs=60, bc_lr=1e-3, log=print):
    """Returns (actor, history). The saved checkpoint holds the best validation score; its "iter" is -1
    if the warm start (the copied rule) was never beaten.

    bc_episodes=0 turns the warm start off (training then starts from a random policy).
    The actor and the critic have separate optimisers and separate gradient clips."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    base_cfg = base_cfg or Config()
    actor, critic = Actor(hidden).to(device), Critic(hidden).to(device)
    opt_c = torch.optim.Adam(critic.parameters(), lr=lr_critic)
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    history, best = [], -1e9
    t_start = time.time()
    final_n = curriculum(iterations - 1, iterations)
    val = lambda pol: validation_score(pol, final_n, base_cfg)
    ref = val(nearest_policy)
    log(f"reference: nearest-task rule scores {ref:.3f} on {len(VALIDATION_SEEDS)} validation farms x "
        f"{len(VALIDATION_SETTINGS)} settings {VALIDATION_SETTINGS} ({final_n} drones; never used for results)")

    meta = {"hidden": hidden, "rule_score": ref}
    if bc_episodes > 0:
        teach = collect_teacher(rng, base_cfg, episodes=bc_episodes)
        opt_bc = torch.optim.Adam(actor.parameters(), lr=bc_lr)
        bc_match, p_relay = pretrain_bc(actor, critic, opt_bc, opt_c, teach, gamma=gamma,
                                        max_epochs=bc_max_epochs, minibatch=minibatch, device=device, rng=rng,
                                        log=log)
        meta.update(bc_match=bc_match, relay_start=p_relay)
        actor.eval()
        best = val(make_policy(actor, device, seed=123))
        torch.save({"actor": actor.state_dict(), "critic": critic.state_dict(), "iter": -1,
                    "val_score": best, **meta}, save_path)
        log(f"   warm-start policy scores {best:.3f} (rule {ref:.3f}) -> saved as the first best: {save_path}")
    opt_a = torch.optim.Adam(actor.parameters(), lr=lr_actor)

    for it in range(iterations):
        n = curriculum(it, iterations)
        rec = Recorder()
        actor.eval()
        sampler = make_policy(actor, device, greedy=False, recorder=rec)
        for _ in range(episodes_per_iter):
            w = World(sample_cfg(rng, n, base_cfg, hard=hard), LearnedSwarm(sampler, recorder=rec))
            w.run()
            rec.finish(w)

        # ---- advantages per decision step
        obs_rows, acts, old_lp, advs, cst, rets = [], [], [], [], [], []
        for ep in rec.episodes:
            if not ep["steps"]:
                continue
            R, disc = segment_rewards(ep, gamma)
            with torch.no_grad():
                F, Mk, G = pad_states([s["cstate"] for s in ep["steps"]], device)
                V = critic(F, Mk, G).cpu().numpy() * VALUE_SCALE
            A, ret = gae(R, disc, V, lam)
            for k, s in enumerate(ep["steps"]):
                n_k = len(s["act"])
                obs_rows.append(s["obs"])
                acts.append(s["act"])
                old_lp.append(s["logp"])
                advs.append(np.full(n_k, A[k], dtype=np.float32))
                cst.append(s["cstate"])
                rets.append(ret[k])
        if not obs_rows:
            continue
        batch = {k: np.concatenate([o[k] for o in obs_rows]) for k in obs_rows[0]}
        acts_np, lp_np, adv_np = np.concatenate(acts), np.concatenate(old_lp), np.concatenate(advs)
        adv_np = (adv_np - adv_np.mean()) / (adv_np.std() + 1e-8)
        rets_np = np.asarray(rets, dtype=np.float32) / VALUE_SCALE

        # ---- PPO update
        actor.train()
        n_dec, n_st = len(acts_np), len(cst)
        for _ in range(epochs):
            perm = rng.permutation(n_dec)
            for b0 in range(0, n_dec, minibatch):
                idx = perm[b0:b0 + minibatch]
                ob = to_t({k: v[idx] for k, v in batch.items()}, device)
                logits, _ = actor(ob)
                dist = torch.distributions.Categorical(logits=logits)
                a = torch.as_tensor(acts_np[idx], device=device)
                lp = dist.log_prob(a)
                ratio = torch.exp(lp - torch.as_tensor(lp_np[idx], device=device))
                ad = torch.as_tensor(adv_np[idx], device=device)
                pg = -torch.min(ratio * ad, torch.clamp(ratio, 1 - clip, 1 + clip) * ad).mean()
                ent = dist.entropy().mean()
                step(opt_a, pg - ent_coef * ent, actor.parameters(), 0.5)
                sidx = rng.choice(n_st, size=min(minibatch, n_st), replace=False)
                F, Mk, G = pad_states([cst[j] for j in sidx], device)
                vl = ((critic(F, Mk, G) - torch.as_tensor(rets_np[sidx], device=device)) ** 2).mean()
                step(opt_c, vl, critic.parameters(), 1.0)

        mk = [ep["metrics"]["makespan"] for ep in rec.episodes]
        fin = np.mean([m is not None for m in mk])
        relay_share = float(np.mean(acts_np == RELAY_ACTION))
        row = {"iter": it, "n_drones": n, "return": float(np.mean([e["return"] for e in rec.episodes])),
               "finished": float(fin), "decisions": n_dec, "relay_share": relay_share,
               "makespan": float(np.mean([m for m in mk if m is not None])) if fin > 0 else None,
               "minutes": (time.time() - t_start) / 60}
        history.append(row)
        log(f"iter {it:3d}  drones {n:2d}  return {row['return']:6.2f}  finished {fin:4.0%}  "
            f"makespan {row['makespan'] if row['makespan'] else float('nan'):6.1f}  relay {relay_share:5.1%}  "
            f"[{row['minutes']:.1f} min]")

        if n == final_n and ((it + 1) % eval_every == 0 or it == iterations - 1):
            actor.eval()
            score = val(make_policy(actor, device, seed=123))
            row["eval_score"] = score
            log(f"   validation: {score:.3f}   (nearest-task rule {ref:.3f}, best so far {best:.3f})")
            if score > best:
                best = score
                torch.save({"actor": actor.state_dict(), "critic": critic.state_dict(), "iter": it,
                            "val_score": score, **meta}, save_path)
                log(f"   saved best policy -> {save_path}")
    if best == -1e9:  # no warm start and never evaluated: save what we have
        torch.save({"actor": actor.state_dict(), "critic": critic.state_dict(), "iter": iterations - 1,
                    "val_score": None, **meta}, save_path)
    return actor, history

In [ ]:
%%writefile swarmnet/experiments.py
"""Run the study: every method on the same farms, across radio ranges, failures and swarm sizes.

Main measure: GAP TO OUR CENTRAL PLAN
    reference ("ideal" in the code) = our central planner with unlimited radio and no failures
    (perfect information, readings reach the base instantly). It is near-optimal on small farms
    (checked against an exact solver) but not proven optimal on 100 points.
    gap = (method's finish time / reference finish time - 1) * 100 %, on the SAME farm (same seed).

Methods
    central   our central planner, run with the real radio range and failures
    nearest   the nearest-task rule, run inside the learned-swarm controller (same plumbing)
    learned   the trained policy, in the same controller
    auction   the auction swarm (CBBA); kept in the code for reference, not used in the notebook
"""
import os
import time

import numpy as np
import pandas as pd

from .central import CentralPlanner
from .config import Config
from .learned import LearnedSwarm, nearest_policy
from .swarm import AuctionSwarm, NearestTask
from .world import World


def make_controller(method, policy=None, central_time=3):
    if method == "central":
        return CentralPlanner(time_limit=central_time)
    if method == "nearest":
        return LearnedSwarm(nearest_policy, name="nearest")
    if method == "learned":
        if policy is None:
            raise ValueError("method 'learned' needs a trained policy (see marl.load_policy)")
        return LearnedSwarm(policy, name="learned")
    if method == "auction":
        return AuctionSwarm()
    if method == "nearest_v13":
        return NearestTask()   # the older stand-alone nearest-task swarm (v1.3 and before)
    raise ValueError(method)


def run_one(method, cfg, policy=None, central_time=3, keep_world=False):
    w = World(cfg, make_controller(method, policy, central_time))
    m = w.run()
    return (m, w) if keep_world else m


def ideal_makespan(cfg, central_time=3):
    """Central planner, unlimited radio, no failures: the best case for this farm."""
    m = run_one("central", cfg.with_(radio_range=1e6, fail_count=0, trace_every=0), central_time=central_time)
    return m["makespan"]


def sweep(methods=("central", "nearest"), radio_ranges=(60, 90, 120, 150, 200, 1e6),
          fail_counts=(0, 1, 3), seeds=range(5), n_drones=10, tasks_per_drone=10, policy=None,
          base_cfg=None, central_time=3, out_csv="results/sweep.csv", log=print):
    base_cfg = base_cfg or Config()
    rows, t0 = [], time.time()
    for s in seeds:
        cfg0 = base_cfg.with_(n_drones=n_drones, n_tasks=tasks_per_drone * n_drones, seed=s, trace_every=0)
        ideal = ideal_makespan(cfg0, central_time)
        for f in fail_counts:
            for r in radio_ranges:
                for meth in methods:
                    m = run_one(meth, cfg0.with_(radio_range=float(r), fail_count=f), policy, central_time)
                    m["ideal"] = ideal
                    m["gap_pct"] = (100.0 * (m["makespan"] / ideal - 1.0)) if m["makespan"] else np.nan
                    rows.append(m)
        log(f"seed {s} done  ({(time.time() - t0) / 60:.1f} min)")
    df = pd.DataFrame(rows)
    if out_csv:
        os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
        df.to_csv(out_csv, index=False)
    return df


def scaling(methods=("nearest",), sizes=(5, 10, 20, 30), seeds=range(3), radio_range=150,
            fail_count=0, policy=None, base_cfg=None, central_time=3, out_csv="results/scaling.csv", log=print):
    """Same work per drone (10 points), growing swarm. For 'learned', the SAME trained policy is used
    at every size: no retraining."""
    base_cfg = base_cfg or Config()
    rows = []
    for n in sizes:
        for s in seeds:
            cfg = base_cfg.with_(n_drones=n, n_tasks=10 * n, seed=s, radio_range=radio_range,
                                 fail_count=fail_count, trace_every=0)
            ideal = ideal_makespan(cfg, central_time)
            for meth in methods:
                t0 = time.time()
                m = run_one(meth, cfg, policy, central_time)
                m["ideal"] = ideal
                m["gap_pct"] = (100.0 * (m["makespan"] / ideal - 1.0)) if m["makespan"] else np.nan
                m["wall_seconds"] = time.time() - t0
                rows.append(m)
        log(f"{n} drones done")
    df = pd.DataFrame(rows)
    if out_csv:
        os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
        df.to_csv(out_csv, index=False)
    return df


def summary(df, by=("method", "radio_range", "fail_count")):
    cols = ["gap_pct", "makespan", "delivered_frac", "mean_lambda2", "repeat_surveys", "messages",
            "recovery_time", "relay_seconds", "readings_lost"]
    cols = [c for c in cols if c in df.columns]
    g = df.groupby(list(by))[cols].agg(["mean", "std"]).round(2)
    return g


def paired_compare(df, a="learned", b="nearest", by=("radio_range", "fail_count"), level=0.95):
    """Method a vs method b on the SAME farms: mean of (gap_a - gap_b), its confidence interval, and
    how often a finished faster. Negative = a is faster. Pairing removes farm-to-farm noise, so this
    is the fair way to ask "did learning beat the rule?"."""
    from scipy import stats
    keys = ["seed", "n_drones"] + list(by)
    keys = [k for k in dict.fromkeys(keys) if k in df.columns]
    A = df[df["method"] == a].set_index(keys)["makespan"]
    B = df[df["method"] == b].set_index(keys)["makespan"]
    j = pd.concat([A.rename("a"), B.rename("b")], axis=1).dropna().reset_index()
    j["diff_pct"] = 100.0 * (j["a"] / j["b"] - 1.0)
    rows = []
    for key, g in j.groupby(list(by)):
        v = g["diff_pct"].to_numpy()
        n = len(v)
        half = float(stats.t.ppf(0.5 + level / 2, n - 1) * v.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan
        key = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(by, key)), "n_farms": n,
                     f"{a}_vs_{b}_pct": round(float(v.mean()), 1),
                     "ci_low": round(float(v.mean()) - half, 1) if n > 1 else np.nan,
                     "ci_high": round(float(v.mean()) + half, 1) if n > 1 else np.nan,
                     f"{a}_faster_on": f"{int((v < 0).sum())}/{n}"})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------- validity checks
def makespan_lower_bound(cfg):
    """A finish time no plan can beat (open routes from the base, no failures).

    Two bounds, take the larger:
      * the farthest point: fly there and survey it;
      * total work shared perfectly: every point is surveyed once and entered by one flight at
        least as long as its nearest neighbour (base or another point), split over all drones.
    """
    w = World(cfg.with_(trace_every=0), NearestTask())
    tasks, base = w.task_xy, w.base
    d_base = np.hypot(*(tasks - base).T) / cfg.speed
    pts = np.vstack([tasks, base[None, :]])
    D = np.hypot(*(tasks[:, None, :] - pts[None, :, :]).transpose(2, 0, 1)) / cfg.speed
    D[np.arange(len(tasks)), np.arange(len(tasks))] = np.inf
    far = float(np.max(d_base + cfg.survey_time))
    share = float((len(tasks) * cfg.survey_time + D.min(axis=1).sum()) / cfg.n_drones)
    return max(far, share)


def check_ideal(cfg, central_time=3, repeats=2):
    """The reference plan must be reproducible and must not beat the lower bound. Returns a dict."""
    vals = [ideal_makespan(cfg, central_time) for _ in range(repeats)]
    lb = makespan_lower_bound(cfg)
    return {"ideal_runs": vals, "reproducible": len(set(vals)) == 1, "lower_bound": round(lb, 1),
            "ideal_over_bound": round(vals[0] / lb, 3), "above_bound": vals[0] >= lb - 1e-6}


def validate(df, tol_pct=0.5):
    """Sanity checks on a sweep or scaling table. Raises AssertionError with a clear message.

      1. no mission may finish faster than our central plan for its farm (beyond tol_pct);
      2. the central planner with unlimited radio and no failures IS the reference, so its gap is 0.
    """
    d = df[df["makespan"].notna()]
    beat = d[d["gap_pct"] < -tol_pct]
    msgs = []
    if len(beat):
        worst = beat.sort_values("gap_pct").iloc[0]
        msgs.append(f"{len(beat)} missions beat our central plan (worst: {worst['method']} at "
                    f"{worst['gap_pct']:.1f}%). The reference plan is not near-optimal: do not use these results.")
    if "radio_range" in d and "fail_count" in d:
        c = d[(d["method"] == "central") & (d["radio_range"] >= 1e5) & (d["fail_count"] == 0)]
        bad = c[c["gap_pct"].abs() > 1e-6]
        if len(bad):
            msgs.append(f"central planner at unlimited radio differs from the reference on {len(bad)} farms "
                        f"(gaps {sorted(bad['gap_pct'].round(1).tolist())}): the planner is not deterministic.")
    if msgs:
        raise AssertionError("RESULTS NOT VALID:\n - " + "\n - ".join(msgs))
    return f"checks passed: {len(d)} missions, none beat our central plan"


def mean_ci(df, by, col="gap_pct", level=0.95):
    """Mean of `col` per group with a t-based confidence interval. With few farms the interval is
    wide: groups whose intervals overlap are not distinguishable."""
    from scipy import stats
    rows = []
    for key, g in df.groupby(list(by)):
        v = g[col].dropna().to_numpy()
        n = len(v)
        half = float(stats.t.ppf(0.5 + level / 2, n - 1) * v.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan
        key = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(by, key)), "n_farms": n, "mean": round(float(v.mean()), 1) if n else np.nan,
                     "ci_low": round(float(v.mean()) - half, 1) if n > 1 else np.nan,
                     "ci_high": round(float(v.mean()) + half, 1) if n > 1 else np.nan})
    return pd.DataFrame(rows)

In [ ]:
%%writefile swarmnet/plots.py
"""Charts for the study (matplotlib, saved as PNG and shown in Colab)."""
import os

import matplotlib.pyplot as plt
import numpy as np

# fixed colour per method (identity never depends on rank); our central plan is a grey reference
COLORS = {"auction": "#2a78d6", "nearest": "#eb6834", "learned": "#1baf7a", "central": "#6b6a64"}
LABELS = {"auction": "Auction swarm (CBBA)", "nearest": "Nearest-task rule",
          "learned": "Learned swarm (GNN + MAPPO, with relay)", "central": "Central planner (real radio)"}
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"


def _style(ax, title, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    ax.figure.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", fontsize=12, color=INK, pad=12)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED)


def _lines(ax, df, x, y, methods):
    for m in methods:
        d = df[df["method"] == m].groupby(x)[y].agg(["mean", "std"]).reset_index()
        if d.empty:
            continue
        xs = np.arange(len(d)) if x == "radio_range" else d[x].values
        ax.errorbar(xs, d["mean"], yerr=d["std"].fillna(0), color=COLORS.get(m, INK), lw=2,
                    marker="o", ms=7, capsize=3, label=LABELS.get(m, m))
        ax.annotate(LABELS.get(m, m).split(" (")[0], (xs[-1], d["mean"].values[-1]), xytext=(6, 0),
                    textcoords="offset points", va="center", fontsize=9, color=MUTED)
    return ax


def _save(fig, path):
    if path:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        fig.savefig(path, dpi=160, bbox_inches="tight")
    return fig


def gap_vs_range(df, fail_count=0, methods=("central", "nearest", "learned"), path=None):
    d = df[df["fail_count"] == fail_count]
    methods = [m for m in methods if m in set(d["method"])]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    _lines(ax, d, "radio_range", "gap_pct", methods)
    ranges = sorted(d["radio_range"].unique())
    ax.set_xticks(range(len(ranges)))
    ax.set_xticklabels(["unlimited" if r >= 1e5 else f"{int(r)} m" for r in ranges])
    ax.axhline(0, color=MUTED, lw=1, ls="--")
    ax.text(0, 0, " our central plan (unlimited radio)", va="bottom", fontsize=9, color=MUTED)
    _style(ax, "How much slower than our central plan, as the radio range changes",
           "Radio range", "Extra time vs our central plan (%)")
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    return _save(fig, path)


def gap_vs_failures(df, radio_range=150, methods=("central", "nearest", "learned"), path=None):
    d = df[df["radio_range"] == radio_range]
    methods = [m for m in methods if m in set(d["method"])]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    _lines(ax, d, "fail_count", "gap_pct", methods)
    ax.set_xticks(sorted(d["fail_count"].unique()))
    _style(ax, f"Cost of drone failures (radio range {int(radio_range)} m)",
           "Drones that fail mid-mission", "Extra time vs our central plan (%)")
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    return _save(fig, path)


def connectivity_vs_range(df, fail_count=0, methods=("central", "nearest", "learned"), path=None):
    d = df[(df["fail_count"] == fail_count) & (df["radio_range"] < 1e5)]
    methods = [m for m in methods if m in set(d["method"])]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    _lines(ax, d, "radio_range", "mean_lambda2", methods)
    ranges = sorted(d["radio_range"].unique())
    ax.set_xticks(range(len(ranges)))
    ax.set_xticklabels([f"{int(r)} m" for r in ranges])
    _style(ax, "Network health during the mission (algebraic connectivity)",
           "Radio range", "Mean algebraic connectivity  (0 = network broken)")
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    return _save(fig, path)


def scaling_plot(df, methods=("nearest", "learned"), path=None):
    methods = [m for m in methods if m in set(df["method"])]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    _lines(ax, df, "n_drones", "gap_pct", methods)
    ax.set_xticks(sorted(df["n_drones"].unique()))
    _style(ax, "Does it still work as the swarm grows? (same policy, no retraining)",
           "Number of drones", "Extra time vs our central plan (%)")
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    return _save(fig, path)


def training_curve(history, path=None):
    it = [h["iter"] for h in history]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(it, [h["return"] for h in history], color=COLORS["learned"], lw=2)
    for n in sorted({h["n_drones"] for h in history}):
        first = next(h["iter"] for h in history if h["n_drones"] == n)
        ax.axvline(first, color=GRID, lw=1)
        ax.text(first, ax.get_ylim()[1], f" {n} drones", va="top", fontsize=9, color=MUTED)
    _style(ax, "Learning curve (team reward per mission)", "Training iteration", "Average return")
    return _save(fig, path)


def eval_curve(history, ref, path=None):
    """Validation score of the policy during training, against the nearest-task rule (dashed)."""
    pts = [(h["iter"], h["eval_score"]) for h in history if h.get("eval_score") is not None]
    fig, ax = plt.subplots(figsize=(8, 4))
    if pts:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], color=COLORS["learned"], lw=2, marker="o", ms=5,
                label="learned policy")
    ax.axhline(ref, color=COLORS["nearest"], lw=1.5, ls="--", label="nearest-task rule")
    _style(ax, "Validation score during training (higher is better)", "Training iteration", "Score")
    ax.legend(frameon=False, fontsize=9, loc="lower right")
    return _save(fig, path)

In [ ]:
%%writefile swarmnet/replay.py
"""Replay dashboard: one self-contained HTML file that plays back a recorded mission.

    from swarmnet.replay import save_html
    save_html(world.trace(), "replays/auction.html")
"""
import json
import os

TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>SwarmNet Replay</title>
<style>
:root{--bg:#fcfcfb;--panel:#f3f2ee;--ink:#0b0b0b;--muted:#52514e;--line:#dddcd6;
 --work:#2a78d6;--home:#6250d6;--idle:#8f8e87;--dead:#e34948;--held:#eda100;--dlv:#1baf7a;--open:#b9b8b0;
 --link:#1baf7a;--link2:#c9c8c0;--base:#0b0b0b}
@media (prefers-color-scheme: dark){:root:not([data-theme="light"]){--bg:#1a1a19;--panel:#242423;--ink:#f0efec;--muted:#c3c2b7;--line:#3a3a38;
 --work:#3987e5;--home:#9085e9;--idle:#8f8e87;--dead:#e66767;--held:#c98500;--dlv:#199e70;--open:#5b5a55;--link:#199e70;--link2:#4a4a47;--base:#f0efec}}
:root[data-theme="dark"]{--bg:#1a1a19;--panel:#242423;--ink:#f0efec;--muted:#c3c2b7;--line:#3a3a38;
 --work:#3987e5;--home:#9085e9;--idle:#8f8e87;--dead:#e66767;--held:#c98500;--dlv:#199e70;--open:#5b5a55;--link:#199e70;--link2:#4a4a47;--base:#f0efec}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:14px/1.45 system-ui,-apple-system,Segoe UI,sans-serif}
.wrap{max-width:1180px;margin:0 auto;padding:20px 16px}
h1{font-size:20px;margin:0 0 4px}.sub{color:var(--muted);margin:0 0 14px}
.stats{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:12px}
.stat{background:var(--panel);border-radius:8px;padding:8px 12px;min-width:110px}
.stat b{display:block;font-size:18px}.stat span{color:var(--muted);font-size:12px}
.grid{display:grid;grid-template-columns:minmax(0,1fr) 300px;gap:16px}
@media (max-width:820px){.grid{grid-template-columns:1fr}}
canvas{width:100%;height:auto;background:var(--panel);border-radius:8px;display:block}
.controls{display:flex;flex-wrap:wrap;align-items:center;gap:10px;margin:10px 0}
button{background:var(--ink);color:var(--bg);border:0;border-radius:6px;padding:7px 14px;font:inherit;cursor:pointer}
input[type=range]{flex:1;min-width:140px}
label{color:var(--muted);font-size:13px;display:flex;align-items:center;gap:4px}
.side{background:var(--panel);border-radius:8px;padding:14px}
.side h2{font-size:14px;margin:0 0 8px}
.leg{display:grid;grid-template-columns:18px 1fr;gap:6px 8px;align-items:center;font-size:13px;margin-bottom:14px}
.dot{width:12px;height:12px;border-radius:50%}
.ln{width:16px;height:0;border-top:2px solid}
.ev{font-size:13px;color:var(--muted);margin:0;padding-left:16px}
table{width:100%;border-collapse:collapse;font-size:13px}td{padding:3px 0;border-bottom:1px solid var(--line)}td:last-child{text-align:right}
</style></head><body><div class="wrap">
<h1 id="title">SwarmNet replay</h1><p class="sub" id="subtitle"></p>
<div class="stats">
 <div class="stat"><b id="s-time">0</b><span>seconds</span></div>
 <div class="stat"><b id="s-dlv">0</b><span>readings delivered</span></div>
 <div class="stat"><b id="s-alive">0</b><span>drones flying</span></div>
 <div class="stat"><b id="s-contact">0</b><span>drones in contact with base</span></div>
 <div class="stat"><b id="s-lam">0</b><span>network health (algebraic connectivity)</span></div>
</div>
<div class="grid"><div>
 <canvas id="cv" width="900" height="900"></canvas>
 <div class="controls">
  <button id="play">Play</button>
  <input type="range" id="scrub" min="0" value="0" aria-label="Time">
  <label>Speed <select id="speed"><option value="1">1x</option><option value="3" selected>3x</option><option value="8">8x</option></select></label>
  <label><input type="checkbox" id="c-links" checked> radio links</label>
  <label><input type="checkbox" id="c-range"> radio range</label>
  <label><input type="checkbox" id="c-tgt" checked> targets</label>
 </div>
</div>
<div class="side">
 <h2>Legend</h2>
 <div class="leg">
  <div class="dot" style="background:var(--work)"></div><div>Drone working (flying to a point or surveying)</div>
  <div class="dot" style="background:var(--home)"></div><div>Drone heading home or recharging</div>
  <div class="dot" style="background:var(--idle)"></div><div>Drone parked at base</div>
  <div class="dot" style="background:var(--dead)"></div><div>Drone failed</div>
  <div class="dot" style="background:var(--open);border:1px solid var(--muted)"></div><div>Point not surveyed yet</div>
  <div class="dot" style="background:var(--held)"></div><div>Surveyed, reading not home yet</div>
  <div class="dot" style="background:var(--dlv)"></div><div>Reading delivered to base</div>
  <div class="ln" style="border-color:var(--link)"></div><div>Radio link connected to base</div>
  <div class="ln" style="border-color:var(--link2);border-top-style:dashed"></div><div>Radio link cut off from base</div>
 </div>
 <h2>Events</h2><ul class="ev" id="events"></ul>
 <h2 style="margin-top:14px">Final result</h2><table id="final"></table>
</div></div></div>
<script>
const TRACE = __TRACE__;
const M = TRACE.meta, F = TRACE.frames;
const cv = document.getElementById('cv'), ctx = cv.getContext('2d');
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const $ = id => document.getElementById(id);
const names = {central:'Central planner',auction:'Auction swarm (CBBA)',nearest:'Nearest-task swarm',learned:'Learned swarm (GNN + MAPPO)'};
$('title').textContent = 'SwarmNet replay: ' + (names[M.method] || M.method);
$('subtitle').textContent = `${M.n_drones} drones, ${M.n_tasks} survey points, radio range ${M.radio_range >= 1e5 ? 'unlimited' : M.radio_range + ' m'}, ${M.field} m field`;
$('scrub').max = F.length - 1;
$('events').innerHTML = M.events.length ? M.events.map(e => `<li>t = ${e.t}s: drone ${e.drone} ${e.what}${e.lost ? ` (${e.lost} readings lost)` : ''}</li>`).join('') : '<li>No failures</li>';
const fm = M.metrics, rows = [['Finished', fm.finished ? 'yes' : 'no'], ['Finish time', fm.makespan ? fm.makespan + ' s' : '-'],
 ['Readings delivered', Math.round(fm.delivered_frac * 100) + '%'], ['Total distance', Math.round(fm.total_distance) + ' m'],
 ['Repeat surveys', fm.repeat_surveys], ['Messages sent', fm.messages], ['Mean connectivity', fm.mean_lambda2.toFixed(2)]];
$('final').innerHTML = rows.map(r => `<tr><td>${r[0]}</td><td>${r[1]}</td></tr>`).join('');

const PAD = 40, S = (cv.width - 2 * PAD) / M.field;
const X = x => PAD + x * S, Y = y => cv.height - PAD - y * S;
let idx = 0, playing = false, acc = 0, last = 0;

function draw() {
  const f = F[idx], W = cv.width, H = cv.height;
  ctx.clearRect(0, 0, W, H);
  ctx.strokeStyle = css('--line'); ctx.lineWidth = 1; ctx.strokeRect(X(0), Y(M.field), M.field * S, M.field * S);
  const nodes = f.d.map(d => [d[0], d[1]]).concat([M.base]);
  const inBase = new Set(f.bc);
  if ($('c-range').checked) {
    ctx.fillStyle = css('--link'); ctx.globalAlpha = 0.06;
    f.d.forEach(d => { if (d[2]) { ctx.beginPath(); ctx.arc(X(d[0]), Y(d[1]), M.radio_range * S, 0, 7); ctx.fill(); } });
    ctx.globalAlpha = 1;
  }
  for (let j = 0; j < M.tasks.length; j++) {
    const s = f.ts[j], t = M.tasks[j];
    ctx.beginPath(); ctx.arc(X(t[0]), Y(t[1]), 4.5, 0, 7);
    if (s === '0') { ctx.fillStyle = css('--open'); ctx.fill(); }
    else { ctx.fillStyle = s === '1' ? css('--held') : css('--dlv'); ctx.fill(); }
  }
  if ($('c-links').checked) {
    for (const [a, b] of f.l) {
      const ok = inBase.has(a) && inBase.has(b);
      ctx.strokeStyle = ok ? css('--link') : css('--link2'); ctx.lineWidth = ok ? 2 : 1.5;
      ctx.setLineDash(ok ? [] : [5, 4]);
      ctx.beginPath(); ctx.moveTo(X(nodes[a][0]), Y(nodes[a][1])); ctx.lineTo(X(nodes[b][0]), Y(nodes[b][1])); ctx.stroke();
    }
    ctx.setLineDash([]);
  }
  ctx.fillStyle = css('--base'); ctx.fillRect(X(M.base[0]) - 9, Y(M.base[1]) - 9, 18, 18);
  ctx.fillStyle = css('--muted'); ctx.font = '13px system-ui'; ctx.fillText('base', X(M.base[0]) + 13, Y(M.base[1]) - 4);
  f.d.forEach((d, i) => {
    const [x, y, alive, mode, held, tgt] = d;
    if ($('c-tgt').checked && alive && tgt) {
      ctx.strokeStyle = css('--muted'); ctx.globalAlpha = 0.35; ctx.lineWidth = 1;
      ctx.beginPath(); ctx.moveTo(X(x), Y(y)); ctx.lineTo(X(tgt[0]), Y(tgt[1])); ctx.stroke(); ctx.globalAlpha = 1;
    }
    const col = !alive ? css('--dead') : (mode === 1 || mode === 2) ? css('--work') : (mode === 3 || mode === 4) ? css('--home') : css('--idle');
    if (!alive) {
      ctx.strokeStyle = col; ctx.lineWidth = 2.5;
      ctx.beginPath(); ctx.moveTo(X(x) - 6, Y(y) - 6); ctx.lineTo(X(x) + 6, Y(y) + 6); ctx.moveTo(X(x) + 6, Y(y) - 6); ctx.lineTo(X(x) - 6, Y(y) + 6); ctx.stroke();
      return;
    }
    ctx.beginPath(); ctx.arc(X(x), Y(y), 8, 0, 7); ctx.fillStyle = col; ctx.fill();
    ctx.lineWidth = 2; ctx.strokeStyle = css('--bg'); ctx.stroke();
    if (held) { ctx.fillStyle = css('--held'); ctx.beginPath(); ctx.arc(X(x) + 8, Y(y) - 8, 4, 0, 7); ctx.fill(); }
  });
  const delivered = (f.ts.match(/2/g) || []).length, alive = f.d.filter(d => d[2]).length;
  $('s-time').textContent = f.t; $('s-dlv').textContent = `${delivered} / ${M.n_tasks}`;
  $('s-alive').textContent = `${alive} / ${M.n_drones}`;
  $('s-contact').textContent = f.bc.filter(n => n < M.n_drones).length;
  $('s-lam').textContent = f.lam2.toFixed(2);
  $('scrub').value = idx;
}
function loop(ts) {
  const dt = last ? (ts - last) / 1000 : 0; last = ts;
  if (playing) { acc += dt * 10 * Number($('speed').value);
    while (acc >= 1) { acc -= 1; if (idx < F.length - 1) idx++; else { playing = false; $('play').textContent = 'Play'; } }
    draw(); }
  requestAnimationFrame(loop);
}
$('play').onclick = () => { if (idx >= F.length - 1) idx = 0; playing = !playing; $('play').textContent = playing ? 'Pause' : 'Play'; };
$('scrub').oninput = e => { idx = Number(e.target.value); draw(); };
['c-links', 'c-range', 'c-tgt'].forEach(id => $(id).onchange = draw);
draw(); requestAnimationFrame(loop);
</script></body></html>
"""


def build_html(trace):
    return TEMPLATE.replace("__TRACE__", json.dumps(trace, separators=(",", ":")))


def save_html(trace, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(build_html(trace))
    return path

## Step 2 — Check the setup

In [ ]:
if USE_ORTOOLS:
    !pip -q install ortools

In [ ]:
import sys, importlib, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, ".")
import swarmnet, torch
from swarmnet.config import Config
from swarmnet.experiments import (run_one, sweep, scaling, ideal_makespan, mean_ci, paired_compare,
                                  check_ideal, validate)
from swarmnet import plots
from swarmnet.replay import save_html
from IPython.display import HTML, display
BASE = Config().with_(use_ortools=USE_ORTOOLS)   # every experiment below starts from this
PREFER = "ortools" if USE_ORTOOLS else "heuristic"
CT = 1 if QUICK else 2                            # central planner search budget
SEEDS = range(2) if QUICK else range(3)           # result farms (validation farms 2000+ are separate)

def show(fig):
    display(fig); plt.close(fig)                  # show a chart exactly once (it is also saved as PNG)

print("swarmnet", swarmnet.__version__, "| PyTorch", torch.__version__, "| central planner:", PREFER)

import os, shutil, datetime
RUN_DIR = f"/content/drive/MyDrive/SwarmNet/v{swarmnet.__version__}_{'quick' if QUICK else 'full'}_{datetime.datetime.now().strftime('%Y-%m-%d_%H%M')}"
if SAVE_TO_DRIVE:   # ask for Google Drive access now, so the long run can save unattended
    from google.colab import drive
    drive.mount("/content/drive")

def backup():
    """Copy everything so far to Google Drive, so a disconnect never loses finished parts."""
    if SAVE_TO_DRIVE:
        for d in ["results", "replays", "runs", "swarmnet"]:
            if os.path.isdir(d):
                shutil.copytree(d, f"{RUN_DIR}/{d}", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
        print("backed up to Google Drive:", RUN_DIR)

## Part 1 — Check our central plan
Every result is measured against our central plan, so it is checked first. The cell **stops the notebook** if a check fails.

* **Exact check:** on 6 small farms (8 points, 3 drones) an exact solver (MILP) finds the true optimum; our plan should match it or come very close.
* **Reproducible:** the same farm must give the same plan every time.
* **Above the lower bound:** no plan can beat a simple bound. The bound is loose, so expect our plan at about 1.4–1.6× it.
* **Not worse as the swarm grows:** with 10 points per drone, the finish time should not grow from 5 to 20 drones.

These show the reference is consistent and near-optimal on small cases; they do not prove it optimal on 100 points, which is why results are called "gap to our central plan".

In [ ]:
from swarmnet.routing import solve, solve_exact, plan_makespan
c = BASE
rng = np.random.default_rng(7)
exact_gaps = []
for trial in range(6):
    pts = rng.uniform(0, c.field_size, size=(8, 2))
    base = np.array(c.base, float)
    starts, offs = [base] * 3, [0.0] * 3
    routes, used = solve(starts, offs, pts, base, c.speed, c.survey_time, prefer=PREFER, time_limit=2)
    fast = plan_makespan(routes, starts, offs, pts, base, c.speed, c.survey_time)
    _, exact, proven = solve_exact(pts, base, 3, c.speed, c.survey_time, time_limit=120)
    exact_gaps.append(100 * (fast / exact - 1))
    print(f"exact check {trial}: our plan {fast:.1f} s | exact {exact:.1f} s "
          f"({'proven optimal' if proven else 'time limit hit'}) | gap {exact_gaps[-1]:.1f}%")
print(f"our plan vs exact optimum on small farms: mean {np.mean(exact_gaps):.1f}%, worst {max(exact_gaps):.1f}%")

ideal_by_size = {}
for n in (5, 10, 20):
    r = check_ideal(BASE.with_(n_drones=n, n_tasks=10 * n, seed=0), central_time=CT)
    ideal_by_size[n] = r["ideal_runs"][0]
    print(f"{n:2d} drones: plan {r['ideal_runs']} s | lower bound {r['lower_bound']} s | "
          f"reproducible {r['reproducible']} | above bound {r['above_bound']}")
    assert r["reproducible"], "STOP: the central plan changes between runs, so gaps are not comparable."
    assert r["above_bound"], "STOP: the central plan beats a lower bound: the simulator or planner has a bug."
growth = ideal_by_size[20] / ideal_by_size[5]
assert growth < 1.3, f"STOP: the central plan grows {growth:.1f}x from 5 to 20 drones with the same work per drone."
print(f"central plan checks passed (20-drone plan / 5-drone plan = {growth:.2f})")

## Part 2 — Train the learned swarm
**Stage 1, warm start (2–4 min).** The network first copies the nearest-task rule on 40 recorded missions until it matches the rule on 98% of decisions. The rule never relays, so "relay" is left out of the copying and then given a 5% starting chance, so the next stage can try it.

**Stage 2, reinforcement learning (MAPPO).** Training missions are mostly **hard** ones (radio 80–150 m, 0–2 drone failures), because that is where the rule is weak. The team earns reward for every survey and every reading brought home, a small bonus for staying connected to the base, and loses a little every second. Training goes 3 → 5 → 10 drones.

**How the best policy is picked.** On 10 validation farms (seeds 2000–2009) in two settings (100 m radio with 1 failure; 150 m with none), never used for the results. The rule's score on the same farms is printed next to it. The warm start is the first candidate (`iter -1`). The cell prints whether the saved policy beats the rule on these farms; Part 3 then tests it on separate farms.

Run the smoke test first (about 1 min). If it errors, stop and copy the error.

In [ ]:
from swarmnet.marl import train, load_policy
_, smoke_hist = train(iterations=2, episodes_per_iter=2, bc_episodes=3, bc_max_epochs=2,
                      save_path="runs/smoke.pt", eval_every=1, device="cpu")
print("SMOKE TEST PASSED")

In [ ]:
ITERS = 40 if QUICK else 80
t0 = time.time()
actor, history = train(iterations=ITERS, episodes_per_iter=6, save_path="runs/learned_policy.pt",
                       device="cpu", eval_every=5 if QUICK else 10)
print(f"training took {(time.time()-t0)/60:.1f} min")
pd.DataFrame(history).to_csv("results/training_history.csv", index=False)
ck = torch.load("runs/learned_policy.pt", map_location="cpu", weights_only=False)
show(plots.training_curve(history, path="results/training_curve.png"))
show(plots.eval_curve(history, ck["rule_score"], path="results/validation_curve.png"))
print(f"warm start matched the rule on {ck['bc_match']:.1%} of decisions")
print(f"saved policy: iteration {ck['iter']} | validation score {ck['val_score']:.3f} vs rule {ck['rule_score']:.3f}")
if ck["val_score"] > ck["rule_score"]:
    print(f"=> on the validation farms the learned policy beats the nearest-task rule "
          f"({ck['val_score']:.3f} vs {ck['rule_score']:.3f}). Part 3 tests this on the result farms.")
else:
    print(f"=> the learned policy did NOT beat the nearest-task rule on the validation farms "
          f"({ck['val_score']:.3f} vs {ck['rule_score']:.3f}). Report this as measured.")
assert ck["bc_match"] >= 0.95, "STOP: the warm start did not learn to copy the rule (< 95%). Send the training log."
policy = load_policy("runs/learned_policy.pt", seed=0)
backup()

## Part 3 — Main study: radio range and drone failures
Every method runs on the same farms. **Gap** = how much slower than our central plan (%). Lower is better. QUICK uses 2 farms per setting (a preview); the reported run uses 3.

The second table is the fair test of learning: **learned vs the rule on the same farms** (paired). Negative = the learned swarm is faster. If the confidence interval crosses 0, the difference is not established.

In [ ]:
if QUICK:
    SW = dict(seeds=SEEDS, radio_ranges=(80, 120, 200, 1e6), fail_counts=(0, 2), central_time=CT)
else:
    SW = dict(seeds=SEEDS, radio_ranges=(80, 120, 150, 200, 1e6), fail_counts=(0, 1, 2), central_time=CT)
t0 = time.time()
df = sweep(methods=("central", "nearest", "learned"), policy=policy, base_cfg=BASE, out_csv="results/sweep.csv", **SW)
print(f"{len(df)} missions in {(time.time()-t0)/60:.1f} min; every mission finished: {df['makespan'].notna().all()}")
print(validate(df))   # stops here if any mission beats our central plan
MID = 120 if QUICK else 150
show(plots.gap_vs_range(df, fail_count=0, path="results/gap_vs_range.png"))
show(plots.gap_vs_failures(df, radio_range=MID, path="results/gap_vs_failures.png"))
tab = df.pivot_table(index=["fail_count", "method"], columns="radio_range", values="gap_pct", aggfunc="mean").round(1)
tab.columns = ["unlimited" if c >= 1e5 else f"{int(c)} m" for c in tab.columns]
print("Mean gap to our central plan (%):"); display(tab)
print("Learned vs nearest-task rule on the same farms (negative = learned faster):")
pc = paired_compare(df, "learned", "nearest"); pc.to_csv("results/learned_vs_rule.csv", index=False); display(pc)
print("Relay use and lost readings (mean per mission):")
display(df.groupby(["method", "fail_count"])[["relay_seconds", "readings_lost", "repeat_surveys"]].mean().round(1))
backup()

## Part 4 — Why failures hurt: the silence timeout
A swarm drone cannot see a failure. It presumes a teammate lost after `fail_detect_time` seconds of silence (90 s by default), and surveyed-but-unsent readings are redone only after 3× that. A shorter timeout recovers faster but can raise false alarms (points surveyed twice when a drone was only out of range). Measured at 100 m and 150 m radio. This is a design choice to report, not a bug.

In [ ]:
rows = []
for fdt in (30, 60, 90):
    d = sweep(methods=("nearest", "learned"), policy=policy, radio_ranges=(100, 150), fail_counts=(0, 1),
              seeds=SEEDS, base_cfg=BASE.with_(fail_detect_time=float(fdt)), central_time=CT, out_csv=None,
              log=lambda *a: None)
    d["fail_detect_time"] = fdt
    rows.append(d)
det = pd.concat(rows, ignore_index=True)
det.to_csv("results/detection_tradeoff.csv", index=False)
backup()
print(validate(det))
det.pivot_table(index=["method", "radio_range", "fail_detect_time"], columns="fail_count",
                values=["gap_pct", "repeat_surveys"], aggfunc="mean").round(1)

## Part 5 — Scaling: more drones, no retraining
Work grows with the swarm (10 points per drone). The learned policy was trained on at most 10 drones and is run here on up to 20 (QUICK) or 30 (reported run) **without retraining**. Read the confidence intervals: with 2 farms per size, QUICK cannot support a claim about scaling.

In [ ]:
SIZES = (5, 10, 20) if QUICK else (5, 10, 20, 30)
t0 = time.time()
sc = scaling(methods=("nearest", "learned"), policy=policy, sizes=SIZES, seeds=SEEDS, radio_range=120,
             fail_count=1, base_cfg=BASE, central_time=CT, out_csv="results/scaling.csv")
print(f"done in {(time.time()-t0)/60:.1f} min"); print(validate(sc))
print("central plan finish time by size (should not grow):", sc.groupby("n_drones")["ideal"].mean().round(0).to_dict())
show(plots.scaling_plot(sc, path="results/scaling.png"))
print("gap to our central plan, mean and 95% confidence interval:"); display(mean_ci(sc, by=("method", "n_drones")))
print("learned vs rule on the same farms:"); display(paired_compare(sc, "learned", "nearest", by=("n_drones",)))
backup()

## Part 6 — Replay one mission
Same farm (seed 0), 120 m radio, one drone fails at t = 150 s. Drones are coloured by what they are doing; radio links are solid when connected to the base and dashed when cut off. A grey drone away from the base is hovering as a relay. All three replays are saved in `replays/`.

In [ ]:
cfg = BASE.with_(radio_range=120, fail_count=1, seed=0)
ideal = ideal_makespan(cfg, central_time=CT)
rows = []
for meth in ["central", "nearest", "learned"]:
    m, w = run_one(meth, cfg, policy=policy, central_time=CT, keep_world=True)
    m["gap_pct"] = 100 * (m["makespan"] / ideal - 1) if m["makespan"] else np.nan
    rows.append(m); save_html(w.trace(), f"replays/{meth}.html")
print(f"our central plan (unlimited radio, no failures) finishes in {ideal:.0f} s")
display(pd.DataFrame(rows)[["method", "makespan", "gap_pct", "delivered_frac", "total_distance", "relay_seconds",
                            "readings_lost", "repeat_surveys", "recovery_time"]].round(1))
display(HTML(open("replays/learned.html").read()))

## Part 7 — Save everything
Creates `swarmnet_results.zip` (all CSVs, charts, replays, the trained policy and the code) and downloads it. With `SAVE_TO_DRIVE = True` it also copies everything to `My Drive/SwarmNet/<version>_<date>/`.

In [ ]:
os.makedirs("export", exist_ok=True)
for d in ["results", "replays", "runs", "swarmnet"]:
    if os.path.isdir(d):
        shutil.copytree(d, f"export/{d}", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
shutil.make_archive("swarmnet_results", "zip", "export")
print("wrote swarmnet_results.zip")
if SAVE_TO_DRIVE:
    shutil.copytree("export", RUN_DIR, dirs_exist_ok=True)
    shutil.copy("swarmnet_results.zip", RUN_DIR)
    print("copied to Google Drive:", RUN_DIR)
try:
    from google.colab import files; files.download("swarmnet_results.zip")
except Exception:
    print("not on Colab: the zip is in", os.getcwd())